# Análise individual — PBIA (sem anexos)

**Projeto:** *As Relações Brasil-China e a Política de Governança Brasileira sobre Inteligência Artificial* (Iniciação Científica FAPESP, Processo nº 2026/04597-1)
**Etapa:** 02 — análise individual, conforme `skill_análise_individual.md` (versão 1.1)
**Insumo:** `PBIA_sem_anexos_extracao.json`, produzido pela skill 01 (extração literal, com as repetições padronizadas marcadas por asterisco)

> ⚠️ Cada visualização indica a sua origem: **visualização prévia** (base inicial para novos gráficos) ou **construída por comando do pesquisador**. **Nada é gravado na pasta `resultados/` sem comando do pesquisador** (`SALVAR_SAIDAS = False`), e nenhum relatório é produzido. A decisão final sobre o que analisar, como e o que manter é sempre do pesquisador.

### 1. Do que se trata
Análise lexical, em português e sem tradução, do texto do documento **Plano Brasileiro de Inteligência Artificial (PBIA), recorte **sem anexos****, a partir do JSON literal da extração. O notebook conta os lemas de conteúdo (unigramas) e os pares de palavras adjacentes (bigramas) e mostra a extensão das unidades e a distribuição dos termos ao longo do texto.

### 2. Finalidade
Conhecer o vocabulário, os enfoques e as ênfases do documento, com métodos explícitos e reprodutíveis, como base para a análise conjunta (skill 03) e para a compreensão da política brasileira de governança da IA.

### 3. Insumo e recorte do conteúdo
O insumo é o recorte **sem anexos** definido pelo pesquisador: páginas 11 a 48 do PDF, da Apresentação ao fim das *Considerações finais*, em **284 unidades** (54 títulos internos, 147 parágrafos e 83 itens de lista). Ficaram fora, na extração, os elementos pré-textuais (p. 1–10), os Anexos 1 e 2 (p. 49–93), os elementos pós-textuais (glossário, siglas e referências, a partir da p. 94), a assinatura da Apresentação, as notas de rodapé, a Figura 1 e as Tabelas 1 e 2. Entram na análise todas as unidades do JSON (títulos internos, parágrafos e itens de lista, de todas as categorias). O título da capa fica fora da contagem, porque a capa está fora do recorte.

### 4. Repetições marcadas e retirada do excesso
Na extração, as palavras que aparecem de forma padronizada e repetida, por seguirem um padrão textual, recebem um asterisco (`palavra*`). Por comando do pesquisador, este notebook **retira só o excesso**: de cada padrão do catálogo, a **primeira ocorrência fica** e é contada como qualquer palavra, e as demais saem da contagem do termo e do total de tokens. Ocorrências sem asterisco nunca são retiradas. **Este JSON não tem padrões marcados**: as repetições padronizadas do PBIA estão nas fichas dos anexos, que não fazem parte deste recorte. O catálogo `repetition_patterns` está vazio e nada é retirado pela regra do excesso marcado.

### 5. Método

| Etapa | O que é feito | Onde se ajusta |
|---|---|---|
| Verificação | Integridade do JSON, do recorte e das marcações; a execução para se algo falhar | — |
| Leitura das marcações | Os asteriscos saem do texto antes do spaCy, e a posição de cada palavra marcada fica guardada | `MARCADOR_REPETICAO` |
| Processamento | Tokenização, classes gramaticais, lemas e frases com o spaCy (`pt_core_news_sm`) | `MODELO_SPACY` |
| Compostos e ênclises | Compostos com hífen ficam como um token (*bem-estar*, *público-privadas*); nas ênclises, o verbo fica separado do pronome (*estima-se* → *estima* + *se*) | `PRONOMES_CLITICOS`, `HIFEN_COMO_TRAVESSAO` |
| Tokens | Palavras do texto: letras, com hífen ou "&" internos (*P&D*). Números, símbolos, pontuação e endereços eletrônicos não são tokens | — |
| Lemas | Minúsculas e com acentos; correções explícitas de lematização | `CORRECOES_DE_LEMA` |
| Conteúdo | Substantivos, nomes próprios, verbos, adjetivos e advérbios fora da lista de *stopwords* do spaCy (verificada pelo lema) e dos modais, com pelo menos 2 caracteres | `CLASSES_DE_CONTEUDO`, `STOPWORDS_ADICIONAIS`, `STOPWORDS_MANTIDAS`, `LEMAS_MODAIS`, `TAMANHO_MINIMO_LEMA` |
| Excesso marcado | Fica a primeira ocorrência de cada padrão; o excesso sai da contagem e do denominador | `RETIRAR_EXCESSO_MARCADO` |
| Medidas | Frequência absoluta, frequência por 1.000 tokens, alcance (unidades), dispersão (DP de Gries, 2008, normalizado por Lijffijt e Gries, 2012) e bigramas (pares de lemas de conteúdo adjacentes, na mesma frase, sem pontuação nem palavra retirada entre eles) | `TOP_N_LEMAS`, `TOP_N_BIGRAMAS`, `TOP_N_DISPERSAO`, `FREQ_MINIMA_BIGRAMA` |

### 6. Transparência das escolhas
- **Comandos do pesquisador aplicados aqui (2026-10-13):** os **30 principais unigramas** e os **25 principais bigramas**, com o recorte **ampliado até o desempate** quando houver empate no ponto de corte (`TOP_N_LEMAS = 30`, `TOP_N_BIGRAMAS = 25`, `AMPLIAR_EMPATES_LEMAS = AMPLIAR_EMPATES_BIGRAMAS = True`). O gráfico marca onde termina o *N* pedido. A instalação do `pt_core_news_sm` foi autorizada pelo pesquisador.
- **Unigramas** são os lemas de conteúdo (seção 3.4 da skill). **Bigramas** são contados pelo par de lemas e exibidos na forma escrita mais frequente no texto (por exemplo, *políticas públicas*); o par de lemas e todas as formas encontradas ficam na tabela. Quando o modelo dá lemas diferentes à mesma forma escrita (*políticas* → *política* ou *político*), as ocorrências dessa forma vão para o par de lemas mais frequente, e a tabela mostra os pares reunidos.
- **Escolhas do agente, visíveis no código e reversíveis por parâmetro:**
  - *Ênclises.* O spaCy lematiza mal a forma inteira (*estima-se* → "estimar se"; *Observando-se* sem lema). O lema do verbo vem de uma passagem auxiliar do spaCy em que só essa forma é reescrita em próclise (*se estima*), sem alterar o texto analisado. As formas truncadas antes de *-lo*, *-la* (*alcançá-los*) recebem o infinitivo por regra fixa (*alcançar*). O perfil do documento lista todas as formas tratadas.
  - *Hífen como travessão.* "FNDCT-não" e "FNDCT-reembolsável" são separadas em palavras (`HIFEN_COMO_TRAVESSAO`), porque o próprio PBIA alterna essas formas com "FNDCT - não reembolsável".
  - *Siglas com "&"* (*P&D*, *PD&I*, *CT&I*) contam como token, como as demais siglas.
  - *Correções de lema* (`CORRECOES_DE_LEMA`, mesma lista nos três notebooks, 287 entradas):
    - (a) lemas inexistentes ou de outra palavra produzidos pelo modelo, sobretudo em formas do subjuntivo e em plurais (*promovam* → "promor", corrigido para *promover*; *vieses* → "viese", corrigido para *viés*; *privadas* → "privar", corrigido para *privado*);
    - (b) pares divididos por maiúscula: em títulos, nomes, inícios de frase e títulos em caixa-alta, o modelo deixa a palavra sem lema (*Ações Estratégicas* → "ações", corrigido para *ação*; *Impactos esperados* → "impactos", corrigido para *impacto*; *SETORES PRODUTIVOS* → "setores", corrigido para *setor*). A palavra recebe o lema que o próprio modelo dá à mesma forma em minúsculas no corpus. Ficam de fora as palavras gramaticais, os componentes de nomes próprios (*Estados Unidos*, *Minas Gerais*), os homógrafos de outra palavra (*São* de São Paulo; *Era* no sentido de época) e *Logística* e *Robótica*, usadas como substantivos;
    - (c) a abreviatura *nº*, cujo lema é *número*.

    Erros de digitação do original (*naciona*, *púbicas*) ficam como estão. O perfil do documento lista as correções aplicadas neste JSON.
- **Depende da decisão do pesquisador:**
  - manter lemas de conteúdo descartados pela lista de *stopwords* (`STOPWORDS_MANTIDAS`; a lista e as frequências estão no perfil do documento);
  - acrescentar *stopwords* (`STOPWORDS_ADICIONAIS`, por exemplo palavras funcionais do inglês);
  - separar *dados* de *dado*;
  - tratar as repetições de conteúdo sem marcação, mostradas na seção própria.

### 7. Princípios
Este notebook segue a skill 02 desta pasta. O comando do pesquisador prevalece sobre qualquer regra e sobre qualquer escolha do agente. O JSON da skill 01 é somente leitura. Nada além do excesso marcado é retirado sem comando do pesquisador. Este notebook não compara documentos nem os dois recortes do PBIA: isso é tarefa da skill 03.


## Configuração
Todos os parâmetros editáveis estão nesta célula. Os parâmetros fixados por comando do pesquisador estão comentados como tal.

In [ ]:
# =============================================================================
# CONFIGURAÇÃO — todos os parâmetros editáveis. Só esta célula muda entre os três notebooks da pasta.
# =============================================================================
SIGLA = "PBIA_sem_anexos"             # "EBIA", "PBIA_sem_anexos" ou "PBIA_com_anexos"
ARQUIVO_JSON = "PBIA_sem_anexos_extracao.json" # JSON da skill 01 (somente leitura)
NOME_SAIDA = "Análise_PBIA_sem_anexos"      # prefixo dos arquivos, se houver gravação
VERSAO_SAIDA = "v1"                         # nova versão a cada mudança; resultados anteriores nunca são sobrescritos
SALVAR_SAIDAS = False                       # nada vai para resultados/ sem comando do pesquisador
SOBRESCREVER = False

# --- Repetições marcadas (skill 02, seção 3.2) ---
MARCADOR_REPETICAO = "*"
RETIRAR_EXCESSO_MARCADO = True              # fica a 1ª ocorrência de cada padrão; sai só o excesso

# --- Recorte do conteúdo (seção 3.1) ---
INCLUIR_TITULO_NA_CONTAGEM = False          # o título vem da capa, que o pesquisador deixou fora do recorte
INCLUIR_TITULOS_INTERNOS = True
CATEGORIAS_EXCLUIDAS = []                   # retirar Foreword, Box ou Annex é decisão do pesquisador
UNIDADES_EXCLUIDAS = {}                     # {id: "motivo"}, só por comando do pesquisador

# --- Pré-processamento (seção 3.4) ---
IDIOMA_ESPERADO = "pt"
MODELO_SPACY = "pt_core_news_sm"            # instalado em 2026-10-13 com autorização do pesquisador
CLASSES_DE_CONTEUDO = ["NOUN", "PROPN", "VERB", "ADJ", "ADV"]
TAMANHO_MINIMO_LEMA = 2
STOPWORDS_ADICIONAIS = []                   # decisão do pesquisador
STOPWORDS_MANTIDAS = []                     # decisão do pesquisador (ver os descartados no perfil do documento)
LEMAS_MODAIS = ["dever", "poder"]
PRONOMES_CLITICOS = ["me", "te", "se", "lhe", "lhes", "nos", "vos", "o", "a", "os", "as",
                     "lo", "la", "los", "las", "no", "na", "nas"]
# Hífen usado como travessão no original: o próprio PBIA alterna "FNDCT-não reembolsável" e
# "FNDCT - não reembolsável". Essas formas são separadas em palavras, e o hífen conta como pontuação.
HIFEN_COMO_TRAVESSAO = ["FNDCT-não", "FNDCT-reembolsável"]
# Correções explícitas de lematização (mesma lista nos três notebooks; ver a seção 6 da Apresentação).
CORRECOES_DE_LEMA = {
    # (a) Lemas inexistentes ou de outra palavra, produzidos pelo lematizador do pt_core_news_sm
    #     (sobretudo formas do subjuntivo, plurais e substantivos lidos como verbo). Chave "lema" vale
    #     para qualquer classe; chave ("lema", "CLASSE") vale só para aquela classe.
    "aborder": "abordar", "adentr": "adentrar", "ademal": "ademais", "advinr": "advir",
    "agilidader": "agilidade", "alcancer": "alcançar", "alhur": "alhures", "almejamo": "almejar",
    "analytic": "analytics", "aprendir": "aprender", "aprendizar": "aprendizado", "apresentir": "apresentar",
    "articuler": "articular", "assegur": "assegurar", "atendar": "atender", "atendir": "atender",
    "atuer": "atuar", "aumentariar": "aumentar", "autônor": "autônomo", "auxiliir": "auxiliar",
    "avalier": "avaliar", "beneficer": "beneficiar", "beneficir": "beneficiar", "cardiopatio": "cardiopatia",
    "cardiopatir": "cardiopatia", "com si": "consigo", "compreendar": "compreender", "compreendir": "compreender",
    "comprometir": "comprometer", "concebir": "conceber", "confir": "conferir", "consultorio": "consultoria",
    "contir": "conter", "contribar": "contribuir", "coordeno": "coordenar", "corrobor": "corroborar",
    "defendar": "defender", "deficiênciar": "deficiência", "demander": "demandar", "denominamor": "denominar",
    "der": "dever", "desenvolr": "desenvolver", "desenvolvir": "desenvolver", "diagnósticar": "diagnóstico",
    "distribuír": "distribuir", "dominir": "dominar", "doutorar": "doutorado", "duradourar": "duradouro",
    "décir": "décimo", "emerger": "emergir", "empatir": "empatia", "empregadore": "empregador",
    "enderecer": "endereçar", "envolr": "envolver", "estabelecir": "estabelecer", "estatísticar": "estatística",
    "exigiar": "exigir", "extensionisto": "extensionista", "extrer": "extremo", "faciliter": "facilitar",
    "ferramentar": "ferramenta", "fomentir": "fomentar", "fornecedoro": "fornecedor", "fornecedora": "fornecedor",
    "fornecir": "fornecer", "forneçar": "fornecer", "fortaleço": "fortalecer", "friser": "frisar",
    "gu": "guiar", "identifiquir": "identificar", "implementir": "implementar", "impulsionir": "impulsionar",
    "incentiver": "incentivar", "incentivir": "incentivar", "incluar": "incluir", "incluír": "incluir",
    "induzar": "induzir", "industriais-tecnológico": "industrial-tecnológico", "influencir": "influenciar",
    "inovadoro": "inovador", "instituír": "instituir", "latte": "lattes", "levarer": "levar",
    "limitir": "limitar", "maximizer": "maximizar", "mereciar": "merecer", "mestrar": "mestrado",
    "minimizir": "minimizar", "miteguir": "mitigar", "mor": "mover", "multifacetar": "multifacetado",
    "métor": "método", "nuvir": "nuvem", "obserr": "observar", "ofereçar": "oferecer", "orientir": "orientar",
    "países-membro": "país-membro", "pedestr": "pedestre", "pilho": "pilha", "possibilito": "possibilitar",
    "possuar": "possuir", "prepar": "preparar", "pressupõer": "pressupor", "previnar": "prevenir",
    "previstar": "prever", "privar": "privado", "produzar": "produzir", "proeminênciar": "proeminência",
    "promor": "promover", "protegir": "proteger", "protejar": "proteger", "raíz": "raiz", "recebir": "receber",
    "redefinar": "redefinir", "refleter": "refletir", "regemr": "reger", "reguladoro": "regulador",
    "resolvir": "resolver", "respeiter": "respeitar", "respeitir": "respeitar", "rápir": "rápido",
    "silvestr": "silvestre", "sobrecargo": "sobrecarga", "spacar": "space", "sub-representada": "sub-representado",
    "substituír": "substituir", "superir": "superar", "trabalhamo": "trabalhar", "trater": "tratar",
    "utilizer": "utilizar", "utilizir": "utilizar", "variávei": "variável", "ventur": "venture",
    "viese": "viés", "vivemo": "viver", "ágel": "ágil",
    "nº": "número",   # abreviatura sem lematização (Decreto nº 9.319/2018); o lema é "número"
    ("autor", "VERB"): "autoria", ("completar", "ADJ"): "completo", ("datar", "NOUN"): "data",
    ("devir", "ADJ"): "devido", ("lógico", "NOUN"): "lógica", ("lógico", "VERB"): "lógica",
    ("métrico", "NOUN"): "métrica", ("político", "NOUN"): "política",
    # (b) Pares divididos por inicial maiúscula: em títulos e nomes, o modelo marca a palavra como nome
    #     próprio e não a lematiza (Ações Estratégicas → "ações"). Regra igual para os três JSONs: a forma
    #     recebe o lema que o próprio modelo dá à mesma forma em minúsculas no corpus. Ficam de fora os
    #     homógrafos de outra palavra ("São" de São Paulo ≠ "são" de ser; "Era" = época ≠ "era" de ser)
    #     e "Logística"/"Robótica", usados como substantivos.
    "abertos": "abrir", "algorítmica": "algorítmico", "aplicada": "aplicar", "aquisições": "aquisição",
    "atende": "atender", "ações": "ação", "básica": "básico", "brasileira": "brasileiro",
    "brasileiros": "brasileiro", "centros": "centro", "cibernética": "cibernético", "ciências": "ciência",
    "científica": "científico", "científicas": "científico", "clara": "claro", "componentes": "componente",
    "comunicações": "comunicação", "dados": "dado", "desafios": "desafio", "diretrizes": "diretriz",
    "direitos": "direito", "econômica": "econômico", "editais": "edital", "educadores": "educador",
    "eixos": "eixo", "empresas": "empresa", "energias": "energia", "energética": "energético",
    "estados": "estado", "estratégicas": "estratégico", "estratégicos": "estratégico", "estudantes": "estudante",
    "estudos": "estudo", "generativa": "generativo", "gerais": "geral", "guias": "guia",
    "hospitalares": "hospitalar", "humanos": "humano", "indicadores": "indicador", "indícios": "indício",
    "informações": "informação", "inovações": "inovação", "institutos": "instituto", "instituições": "instituição",
    "integrado": "integrar", "interdisciplinares": "interdisciplinar", "investimentos": "investimento",
    "juristas": "jurista", "laboratórios": "laboratório", "licitações": "licitação", "materiais": "material",
    "melhor": "bom", "máquinas": "máquina", "nacionais": "nacional", "naturais": "natural", "nações": "nação",
    "normas": "norma", "nova": "novo", "novas": "novo", "objetivos": "objetivo", "orientações": "orientação",
    "parcerias": "parceria", "pequenas": "pequeno", "pessoais": "pessoal", "pessoas": "pessoa",
    "políticas": "política", "projetos": "projeto", "pública": "público", "públicos": "público",
    "recursos": "recurso", "redes": "rede", "relatórios": "relatório", "relações": "relação",
    "renováveis": "renovável", "responsabilidades": "responsabilidade", "serviços": "serviço",
    "sistemas": "sistema", "sociais": "social", "soluções": "solução", "talentos": "talento",
    "tecnologias": "tecnologia", "tecnológica": "tecnológico", "tecnológicas": "tecnológico",
    ("técnica", "PROPN"): "técnico", "técnicas": "técnico", "todos": "todo", "unidades": "unidade", "vale": "valer",
    "áreas": "área", "éticas": "ético",
    # (b, cont.) Palavras com maiúscula no início de frase ou em títulos em caixa-alta (7. APLICAÇÃO NOS SETORES
    #     PRODUTIVOS), que o modelo também deixa sem lema. Só palavras de conteúdo; as gramaticais (Ao, Dessa, Foi)
    #     já são tratadas pela lista de stopwords, e componentes de nomes próprios (Estados Unidos) ficam como estão.
    "abrange": "abranger", "acredita": "acreditar", "almejamos": "almejar", "anomalias": "anomalia", "anos": "ano",
    "aspectos": "aspecto", "assistentes": "assistente", "bases": "base", "brasileiras": "brasileiro",
    "buscamos": "buscar", "centrada": "centrar", "ciclos": "ciclo", "considerações": "consideração",
    "cooperativa": "cooperativo", "dada": "dar", "destaca": "destacar", "envolve": "envolver", "erros": "erro",
    "existe": "existir", "existem": "existir", "exemplos": "exemplo", "extraordinários": "extraordinário",
    "focadas": "focar", "fomenta": "fomentar", "fundamentada": "fundamentar", "imagens": "imagem",
    "impactos": "impacto", "inclui": "incluir", "indústrias": "indústria", "iniciativas": "iniciativa",
    "internacionais": "internacional", "janelas": "janela", "juntos": "junto", "maior": "grande",
    "mecanismos": "mecanismo", "oficiais": "oficial", "organizações": "organização", "orientada": "orientar",
    "padrões": "padrão", "princípios": "princípio", "priorizando": "priorizar", "produtivos": "produtivo",
    "qualificações": "qualificação", "queremos": "querer", "resultados": "resultado", "revoluções": "revolução",
    "sabemos": "saber", "setores": "setor", "temáticos": "temático", "traz": "trazer", "verticais": "vertical",
    "visamos": "visar",
}

# --- Visualizações (seção 3.8) ---
COR_DOCUMENTO = "#2E9E4F"                   # EBIA #00441B; PBIA sem anexos #2E9E4F; PBIA com anexos #A1D99B
TOP_N_LEMAS = 30                            # comando do pesquisador (2026-10-13): os 30 principais unigramas
AMPLIAR_EMPATES_LEMAS = True                # comando do pesquisador: em caso de empate, ampliar até o desempate
TOP_N_BIGRAMAS = 25                         # comando do pesquisador (2026-10-13): os 25 principais bigramas
AMPLIAR_EMPATES_BIGRAMAS = True             # comando do pesquisador: em caso de empate, ampliar até o desempate
FREQ_MINIMA_BIGRAMA = 2
TOP_N_DISPERSAO = 15                        # padrão da skill (desempate alfabético, informado na síntese)


## Ambiente, estilo e funções
Restrição de pasta, pasta de saída versionada, estilo das figuras e funções do pipeline. O código é **idêntico nos três notebooks** da pasta.

In [ ]:
# =============================================================================
# 3. AMBIENTE, ESTILO E FUNÇÕES — código idêntico nos três notebooks da pasta
# =============================================================================
import hashlib
import json
import math
import platform
import re
import sys
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import spacy
from IPython.display import Markdown, display
from matplotlib.ticker import FuncFormatter
from spacy.lang.pt.stop_words import STOP_WORDS as STOPWORDS_SPACY

# --- Restrição de pasta (skill 02, seção 1.3) -------------------------------
PASTA = Path.cwd()
if PASTA.name != "EBIA_PBIA":
    raise RuntimeError(f"Execute este notebook de dentro da pasta EBIA_PBIA (pasta atual: {PASTA}).")
PASTA_SAIDA = PASTA / "resultados" / SIGLA / VERSAO_SAIDA   # usada só com SALVAR_SAIDAS = True
SAIDAS_GRAVADAS = []
np.random.seed(0)   # nenhuma etapa usa sorteio; a semente fica fixada por reprodutibilidade

ROTULO = {"EBIA": "EBIA", "PBIA_sem_anexos": "PBIA (sem anexos)", "PBIA_com_anexos": "PBIA (com anexos)"}[SIGLA]
CUT_ESPERADO = {"EBIA": None, "PBIA_sem_anexos": "sem_anexos", "PBIA_com_anexos": "com_anexos"}[SIGLA]
NOMES_TIPO = {"heading": "Títulos internos", "paragraph": "Parágrafos", "list_item": "Itens de lista"}

# --- Estilo das figuras (fonte ≥ 9 pt; PNG 300 dpi e PDF só com comando de gravação)
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.fontsize": 9, "figure.dpi": 110,
    "savefig.dpi": 300, "axes.spines.top": False, "axes.spines.right": False,
})
COR_CONTORNO = "#00441B"   # contorno escuro, para o tom claro continuar legível
_r, _g, _b = (int(COR_DOCUMENTO[i:i + 2], 16) / 255 for i in (1, 3, 5))
COR_CONTRASTE = "white" if 0.2126 * _r + 0.7152 * _g + 0.0722 * _b < 0.45 else "black"   # mediana sobre a caixa


# --- Números no padrão brasileiro (o valor nunca é alterado, só a escrita) ----
def br(valor, casas=0):
    if valor is None or (isinstance(valor, float) and math.isnan(valor)):
        return "–"
    texto = f"{valor:,.{casas}f}"
    return texto.replace(",", " ").replace(".", ",").replace(" ", ".")


def br_pct(fracao, casas=1):
    return f"{br(100 * fracao, casas)}%"


EIXO_BR = FuncFormatter(lambda v, _: br(v, 0) if abs(v - round(v)) < 1e-9 else br(v, 1))


def tabela_br(df, casas=None):
    """Cópia da tabela com os números escritos no padrão brasileiro, só para exibição."""
    casas = casas or {}
    out = df.copy()
    for col in out.columns:
        serie = out[col]
        if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
            continue
        c = casas.get(col, 0 if pd.api.types.is_integer_dtype(serie) else 2)
        out[col] = serie.map(lambda v: br(v, c))
    return out


def salvar_figura(fig, nome):
    """Grava PNG (300 dpi) e PDF em resultados/<SIGLA>/<VERSAO_SAIDA>/ — só com SALVAR_SAIDAS = True."""
    if not SALVAR_SAIDAS:
        return
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    for ext in ("png", "pdf"):
        alvo = PASTA_SAIDA / f"{NOME_SAIDA}_{nome}.{ext}"
        if alvo.exists() and not SOBRESCREVER:
            raise FileExistsError(f"{alvo} já existe: use uma nova VERSAO_SAIDA.")
        fig.savefig(alvo, dpi=300, bbox_inches="tight")
        SAIDAS_GRAVADAS.append(str(alvo.relative_to(PASTA)))


def salvar_tabela(df, nome):
    """Grava a tabela completa (CSV, UTF-8, separador ';', vírgula decimal) — só com SALVAR_SAIDAS = True."""
    if not SALVAR_SAIDAS:
        return
    PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    alvo = PASTA_SAIDA / f"{NOME_SAIDA}_{nome}.csv"
    if alvo.exists() and not SOBRESCREVER:
        raise FileExistsError(f"{alvo} já existe: use uma nova VERSAO_SAIDA.")
    df.to_csv(alvo, sep=";", decimal=",", encoding="utf-8", index=False)
    SAIDAS_GRAVADAS.append(str(alvo.relative_to(PASTA)))


# --- Passo 0: leitura das marcações (skill 02, seção 3.2) ---------------------
RX_MARCADA = re.compile(r"([^\W\d_]+(?:-[^\W\d_]+)*)" + re.escape(MARCADOR_REPETICAO))


def ler_marcacoes(texto):
    """Tira os asteriscos e devolve o texto literal e (início, fim, palavra) de cada palavra marcada."""
    partes, marcadas, ultimo, tamanho = [], [], 0, 0
    for m in RX_MARCADA.finditer(texto):
        trecho = texto[ultimo:m.start()]
        partes.append(trecho)
        tamanho += len(trecho)
        marcadas.append((tamanho, tamanho + len(m.group(1)), m.group(1)))
        partes.append(m.group(1))
        tamanho += len(m.group(1))
        ultimo = m.end()
    partes.append(texto[ultimo:])
    limpo = "".join(partes)
    if MARCADOR_REPETICAO in limpo:
        raise ValueError(f"Marcador de repetição fora de palavra: {texto[:80]}")
    return limpo, marcadas


def agrupar_ocorrencias(limpo, marcadas):
    """Uma ocorrência = palavras marcadas seguidas, separadas apenas por um espaço (como na skill 01)."""
    grupos = []
    for ini, fim, palavra in marcadas:
        if grupos and limpo[grupos[-1][-1][1]:ini] == " ":
            grupos[-1].append((ini, fim, palavra))
        else:
            grupos.append([(ini, fim, palavra)])
    return grupos


def sha256_arquivo(caminho):
    return hashlib.sha256(Path(caminho).read_bytes()).hexdigest()


def carregar_e_verificar(caminho):
    """Lê o JSON da skill 01 e confere integridade e marcações (skill 02, seção 4.2, item 4)."""
    if not caminho.exists():
        raise FileNotFoundError(f"{caminho.name} não existe: a skill 01 (extração) precisa ser executada antes.")
    js = json.loads(caminho.read_text(encoding="utf-8"))
    doc, ext, secoes = js["document"], js["extraction"], js["sections"]
    catalogo = {p["id"]: p for p in ext.get("repetition_patterns", [])}
    linhas = []

    def checar(nome, ok, detalhe=""):
        linhas.append((nome, "ok" if ok else "FALHOU", detalhe))

    checar("produzido pela skill 01 deste corpus",
           ext.get("skill") == "skill_extração" and ext.get("corpus") == "EBIA_PBIA",
           f"skill = {ext.get('skill')}; corpus = {ext.get('corpus')}")
    checar("recorte corresponde à SIGLA", ext.get("cut") == CUT_ESPERADO, f"cut = {ext.get('cut')}")
    checar("ids sequenciais", [s["id"] for s in secoes] == list(range(1, len(secoes) + 1)), f"1 a {len(secoes)}")
    checar("nenhuma unidade vazia", all(s["text"].strip() for s in secoes))
    checar("número de unidades = registrado", len(secoes) == ext.get("units_extracted"),
           f"{len(secoes)} unidades; registrado {ext.get('units_extracted')}")
    checar("título presente", bool(doc.get("title")), doc.get("title") or "")
    checar("idioma = " + IDIOMA_ESPERADO, doc.get("language") == IDIOMA_ESPERADO, f"language = {doc.get('language')}")
    total_marcadas, grupos_ok, ids_ok, por_padrao = 0, True, True, Counter()
    for s in secoes:
        limpo, marcadas = ler_marcacoes(s["text"])
        total_marcadas += len(marcadas)
        grupos = agrupar_ocorrencias(limpo, marcadas)
        grupos_ok &= len(grupos) == len(s["marked_patterns"])
        for p in s["marked_patterns"]:
            ids_ok &= p in catalogo
            por_padrao[p] += 1
    checar("palavras marcadas = extraction.marked_words", total_marcadas == ext.get("marked_words"),
           f"{total_marcadas} no texto; registrado {ext.get('marked_words')}")
    checar("ocorrências marcadas de cada unidade = marked_patterns", grupos_ok)
    checar("todo padrão usado existe no catálogo", ids_ok, f"{len(catalogo)} padrões no catálogo")
    checar("ocorrências por padrão = catálogo",
           all(por_padrao[i] == p["occurrences"] for i, p in catalogo.items()) and set(por_padrao) <= set(catalogo))
    return js, pd.DataFrame(linhas, columns=["verificação", "resultado", "detalhe"])


# --- Passos 1 a 6: spaCy, ênclises, tokens, lemas e classes (skill 02, seção 3.4)
CLITICOS = sorted(set(PRONOMES_CLITICOS), key=len, reverse=True)
RX_ENCLISE = re.compile(r"([^\W\d_]+)-(" + "|".join(CLITICOS) + r")")
RX_MESOCLISE = re.compile(r"([^\W\d_]+)-(" + "|".join(CLITICOS) + r")-([^\W\d_]+)")
TRUNCADAS = {"á": "ar", "ê": "er", "í": "ir", "ô": "or"}   # alcançá-los → alcançar; fazê-lo → fazer
RX_PALAVRA = re.compile(r"[^\W\d_]+(?:[-&][^\W\d_]+)*")      # letras, com hífen ou & internos (P&D)
STOPWORDS = (set(STOPWORDS_SPACY) | set(STOPWORDS_ADICIONAIS)) - set(STOPWORDS_MANTIDAS)


def tokens_especiais(doc):
    """Ênclises e mesóclises (verbo separado do pronome) e hífens usados como travessão."""
    especiais, trocas = {}, []
    for t in doc:
        m = RX_MESOCLISE.fullmatch(t.text)
        if m:
            raiz, cl, fim = m.groups()
            especiais[t.i] = ("mesóclise", raiz + fim, cl)
            trocas.append((t.idx, t.idx + len(t.text), f"{cl} {(raiz + fim).lower()}".ljust(len(t.text))))
            continue
        m = RX_ENCLISE.fullmatch(t.text)
        if m:
            verbo, cl = m.groups()
            if cl in ("lo", "la", "los", "las") and verbo[-1].lower() in TRUNCADAS:
                especiais[t.i] = ("ênclise (forma truncada)", verbo, cl)
            else:
                especiais[t.i] = ("ênclise", verbo, cl)
                trocas.append((t.idx, t.idx + len(t.text), f"{cl} {verbo.lower()}"))
            continue
        if t.text in HIFEN_COMO_TRAVESSAO:
            especiais[t.i] = ("hífen como travessão", t.text.split("-"), None)
            trocas.append((t.idx, t.idx + len(t.text), t.text.replace("-", " ")))
    return especiais, trocas


def texto_auxiliar(texto, trocas):
    """Texto da passagem auxiliar: mesmo comprimento, só os trechos trocados mudam."""
    pedacos, ultimo = [], 0
    for ini, fim, novo in sorted(trocas):
        assert len(novo) == fim - ini
        pedacos += [texto[ultimo:ini], novo]
        ultimo = fim
    pedacos.append(texto[ultimo:])
    return "".join(pedacos)


def novo_item(unidade, texto, lema, upos, sent, ini=None, origem=None):
    return {"unidade": unidade, "texto": texto, "lema_modelo": lema, "upos": upos,
            "sent": sent, "ini": ini, "origem": origem}


def itens_da_unidade(uid, doc, nlp, registro):
    """Sequência de itens (palavras e não palavras) de uma unidade, na ordem do texto."""
    especiais, trocas = tokens_especiais(doc)
    aux = {}
    if trocas:
        aux = {x.idx: x for x in nlp(texto_auxiliar(doc.text, trocas))}
    itens = []
    for t in doc:
        if t.is_space:
            continue
        sent = t.sent.start
        if t.i not in especiais:
            itens.append(novo_item(uid, t.text, (t.lemma_ or t.text).lower(), t.pos_, sent, ini=t.idx))
            continue
        tipo, a, b = especiais[t.i]
        if tipo in ("ênclise", "mesóclise"):
            v = aux.get(t.idx + len(b) + 1)
            if v is None:
                raise RuntimeError(f"Ênclise não localizada na passagem auxiliar: {t.text} (unidade {uid})")
            upos = v.pos_ if v.pos_ in ("VERB", "AUX") else "VERB"
            itens.append(novo_item(uid, a, v.lemma_.lower(), upos, sent, origem=tipo))
            itens.append(novo_item(uid, b, b.lower(), "PRON", sent, origem=tipo))
            registro.append((t.text, tipo, a, v.lemma_.lower(), v.pos_, b))
        elif tipo == "ênclise (forma truncada)":
            lema = a[:-1].lower() + TRUNCADAS[a[-1].lower()]
            itens.append(novo_item(uid, a, lema, "VERB", sent, origem=tipo))
            itens.append(novo_item(uid, b, b.lower(), "PRON", sent, origem=tipo))
            registro.append((t.text, tipo, a, lema, "VERB (regra)", b))
        else:  # hífen como travessão: cada parte vira palavra; o hífen fica como pontuação
            pos = t.idx
            for k, parte in enumerate(a):
                x = aux.get(pos)
                if x is None or x.text != parte:
                    raise RuntimeError(f"Parte não localizada na passagem auxiliar: {t.text} (unidade {uid})")
                if k:
                    itens.append(novo_item(uid, "-", "-", "PUNCT", sent, origem=tipo))
                itens.append(novo_item(uid, parte, x.lemma_.lower(), x.pos_, sent, origem=tipo))
                pos += len(parte) + 1
            registro.append((t.text, tipo, " + ".join(a), "", "", ""))
    return itens


def processar(js, nlp):
    """Aplica os filtros, o spaCy, a retirada do excesso marcado e as classes de conteúdo."""
    secoes = js["sections"]
    unidades = []
    if INCLUIR_TITULO_NA_CONTAGEM:
        titulo = " ".join(x for x in (js["document"]["title"], js["document"].get("subtitle")) if x)
        unidades.append({"id": 0, "type": "title", "category": "Title", "path": [], "text": titulo, "marked_patterns": []})
    unidades += secoes
    filtros = []
    incluidas = []
    for s in unidades:
        motivo = None
        if s["type"] == "heading" and not INCLUIR_TITULOS_INTERNOS:
            motivo = "títulos internos excluídos (INCLUIR_TITULOS_INTERNOS = False)"
        elif s["category"] in CATEGORIAS_EXCLUIDAS:
            motivo = f"categoria {s['category']} excluída (CATEGORIAS_EXCLUIDAS)"
        elif s["id"] in UNIDADES_EXCLUIDAS:
            motivo = f"unidade excluída: {UNIDADES_EXCLUIDAS[s['id']]}"
        (filtros if motivo else incluidas).append((s, motivo))
    incluidas = [s for s, _ in incluidas]
    limpos, grupos = {}, {}
    for s in incluidas:
        limpo, marcadas = ler_marcacoes(s["text"])
        limpos[s["id"]] = limpo
        grupos[s["id"]] = agrupar_ocorrencias(limpo, marcadas)
    # retirada do excesso: a 1ª ocorrência de cada padrão (na ordem dos id) fica; as demais saem
    ocorrencias, vistos = [], {}
    for s in incluidas:
        for k, (padrao, grupo) in enumerate(zip(s["marked_patterns"], grupos[s["id"]])):
            excesso = RETIRAR_EXCESSO_MARCADO and padrao in vistos
            vistos.setdefault(padrao, s["id"])
            ocorrencias.append({"padrao": padrao, "unidade": s["id"], "ordem": k, "excesso": excesso,
                                "palavras": [p for _, _, p in grupo], "spans": [(i, f) for i, f, _ in grupo]})
    marcada_em = {}
    for o in ocorrencias:
        for ini, fim in o["spans"]:
            marcada_em[(o["unidade"], ini)] = o
    registro_especiais, itens = [], []
    docs = nlp.pipe([limpos[s["id"]] for s in incluidas], batch_size=32)
    for s, doc in zip(incluidas, docs):
        its = itens_da_unidade(s["id"], doc, nlp, registro_especiais)
        localizadas = 0
        for it in its:
            o = marcada_em.get((s["id"], it["ini"])) if it["ini"] is not None else None
            it["padrao"] = o["padrao"] if o else None
            it["excesso"] = bool(o and o["excesso"])
            if o:
                if it["texto"] not in o["palavras"]:
                    raise RuntimeError(f"Palavra marcada não corresponde ao token: unidade {s['id']}, {it['texto']}")
                localizadas += 1
        esperadas = sum(len(g) for g in grupos[s["id"]])
        if localizadas != esperadas:
            raise RuntimeError(f"Unidade {s['id']}: {esperadas} palavras marcadas, {localizadas} localizadas no spaCy.")
        itens += its
    for it in itens:
        it["palavra"] = bool(RX_PALAVRA.fullmatch(it["texto"]))
        it["lema"] = CORRECOES_DE_LEMA.get((it["lema_modelo"], it["upos"]), CORRECOES_DE_LEMA.get(it["lema_modelo"], it["lema_modelo"]))
        it["corrigido"] = it["lema"] != it["lema_modelo"]
        it["stopword"] = it["lema"] in STOPWORDS
        it["modal"] = it["lema"] in LEMAS_MODAIS and it["upos"] in ("VERB", "AUX")
        it["classe_conteudo"] = it["upos"] in CLASSES_DE_CONTEUDO
        it["conteudo_sem_retirada"] = (it["palavra"] and it["classe_conteudo"] and not it["stopword"]
                                       and not it["modal"] and len(it["lema"]) >= TAMANHO_MINIMO_LEMA)
        it["conta"] = it["palavra"] and not it["excesso"]
        it["conteudo"] = it["conteudo_sem_retirada"] and not it["excesso"]
    tab = pd.DataFrame(itens)
    tab["posicao"] = tab["conta"].cumsum().where(tab["conta"])   # posição do token no documento (1..N)
    return {"incluidas": incluidas, "filtros": filtros, "limpos": limpos, "ocorrencias": ocorrencias,
            "itens": tab, "especiais": registro_especiais}


# --- Medidas (skill 02, seção 3.5) ------------------------------------------
def tabela_lemas(R):
    """Frequência, taxa por 1.000 tokens, alcance e DP normalizado de todos os lemas de conteúdo."""
    t = R["itens"]
    n_tokens = int(t["conta"].sum())
    n_antes = int(t["palavra"].sum())
    por_unidade = t[t["conta"]].groupby("unidade").size()
    partes = por_unidade[por_unidade > 0]
    s = partes / partes.sum()
    s_min = float(s.min())
    cont = t[t["conteudo"]]
    linhas = []
    freq_antes = t[t["conteudo_sem_retirada"]].groupby("lema").size()
    for lema, g in cont.groupby("lema"):
        f_u = g.groupby("unidade").size()
        v = f_u / f_u.sum()
        diff = (v - s.reindex(v.index)).abs().sum() + (1 - s.reindex(v.index).sum())
        dp = 0.5 * diff
        linhas.append({"lema": lema, "frequência": len(g), "por_mil": len(g) / n_tokens * 1000,
                       "alcance_unidades": len(f_u), "alcance_pct": len(f_u) / len(partes),
                       "DP_norm": dp / (1 - s_min),
                       "frequência_antes": int(freq_antes.get(lema, 0)),
                       "por_mil_antes": freq_antes.get(lema, 0) / n_antes * 1000})
    df = pd.DataFrame(linhas)
    df["retiradas_excesso"] = df["frequência_antes"] - df["frequência"]
    return df.sort_values(["frequência", "lema"], ascending=[False, True]).reset_index(drop=True)


def bigramas(R, sem_retirada=False):
    """Pares de tokens de conteúdo adjacentes no texto, na mesma frase, sem pontuação nem palavra retirada entre eles.
    Ocorrências com a mesma forma escrita ficam no mesmo bigrama: quando o modelo dá lemas diferentes à mesma forma
    (políticas → política ou político), elas vão para o par de lemas mais frequente entre os dessa forma."""
    t = R["itens"]
    chave = "conteudo_sem_retirada" if sem_retirada else "conteudo"
    ocorrencias = []
    registros = t.to_dict("records")
    for a, b in zip(registros, registros[1:]):
        if a["unidade"] != b["unidade"] or a["sent"] != b["sent"] or not (a[chave] and b[chave]):
            continue
        ocorrencias.append(((a["lema"], b["lema"]), f"{a['texto'].lower()} {b['texto'].lower()}", a["unidade"]))
    total_par = Counter(k for k, _, _ in ocorrencias)
    por_forma = defaultdict(Counter)
    for k, forma, _ in ocorrencias:
        por_forma[forma][k] += 1
    canonico = {forma: sorted(c, key=lambda k: (-total_par[k], k))[0] for forma, c in por_forma.items()}
    pares, formas, unidades, reunidos = Counter(), defaultdict(Counter), defaultdict(set), defaultdict(Counter)
    for k, forma, u in ocorrencias:
        kk = canonico[forma]
        pares[kk] += 1
        formas[kk][forma] += 1
        unidades[kk].add(u)
        reunidos[kk][k] += 1
    return pares, formas, unidades, reunidos


def tabela_bigramas(R):
    n_tokens = int(R["itens"]["conta"].sum())
    pares, formas, unidades, reunidos = bigramas(R)
    pares_antes, formas_antes, _, _ = bigramas(R, sem_retirada=True)
    antes_por_forma = Counter()
    for k, c in formas_antes.items():
        antes_por_forma.update(c)
    linhas = []
    for k, f in pares.items():
        variantes = sorted(formas[k].items(), key=lambda kv: (-kv[1], kv[0]))
        outros = [f"{a} + {b} ({br(n)})" for (a, b), n in reunidos[k].items() if (a, b) != k]
        linhas.append({"bigrama": variantes[0][0], "lemas": f"{k[0]} + {k[1]}", "frequência": f,
                       "por_mil": f / n_tokens * 1000, "alcance_unidades": len(unidades[k]),
                       "frequência_antes": sum(antes_por_forma[x] for x in formas[k]),
                       "formas_no_texto": "; ".join(f"{x} ({br(n)})" for x, n in variantes),
                       "lemas_reunidos": "; ".join(outros) if outros else "–"})
    df = pd.DataFrame(linhas)
    df["retiradas_excesso"] = df["frequência_antes"] - df["frequência"]
    return df.sort_values(["frequência", "bigrama"], ascending=[False, True]).reset_index(drop=True)


def recorte(df, n, coluna="frequência", ampliar_empates=False, chave="lema"):
    """Recorte 'top N'. Com ampliar_empates, inclui todos os empatados com o N-ésimo (comando do pesquisador);
    sem ele, desempate alfabético, com a lista dos empatados que ficaram de fora."""
    if len(df) <= n:
        return df.copy(), {"pedido": n, "exibidos": len(df), "acrescentados": [], "fora_empatados": [], "primeiro_fora": None}
    limite = df.iloc[n - 1][coluna]
    if ampliar_empates:
        sel = df[df[coluna] >= limite]
        acrescentados = list(sel.iloc[n:][chave])
        fora_empatados = []
    else:
        sel = df.iloc[:n]
        acrescentados = []
        fora_empatados = list(df.iloc[n:][df.iloc[n:][coluna] == limite][chave])
    resto = df.iloc[len(sel):]
    primeiro_fora = (resto.iloc[0][chave], resto.iloc[0][coluna]) if len(resto) else None
    return sel.copy(), {"pedido": n, "exibidos": len(sel), "limite": limite, "acrescentados": acrescentados,
                        "fora_empatados": fora_empatados, "primeiro_fora": primeiro_fora}


def barras_horizontais(df, rotulo, valor, titulo, xlabel, info, nome_figura, anotar_por_mil=True):
    n = len(df)
    fig, ax = plt.subplots(figsize=(8.6, max(3.2, 0.27 * n + 1.6)))
    y = np.arange(n)[::-1]
    ax.barh(y, df[valor], color=COR_DOCUMENTO, edgecolor=COR_CONTORNO, linewidth=0.6, height=0.72)
    ax.set_yticks(y)
    ax.set_yticklabels(df[rotulo])
    ax.set_xlim(0, df[valor].max() * 1.22)
    ax.xaxis.set_major_formatter(EIXO_BR)
    for yi, (_, r) in zip(y, df.iterrows()):
        txt = br(r[valor]) + (f"  ({br(r['por_mil'], 2)} por mil)" if anotar_por_mil else "")
        ax.text(r[valor] + df[valor].max() * 0.01, yi, txt, va="center", fontsize=9)
    if info["exibidos"] > info["pedido"]:
        corte = y[info["pedido"] - 1] - 0.5
        ax.axhline(corte, color="0.25", linestyle="--", linewidth=1)
        ax.text(ax.get_xlim()[1], corte - 0.15, f"fim do recorte pedido (N = {info['pedido']}); abaixo, empatados",
                ha="right", va="top", fontsize=9, color="0.25")
    ax.set_title(titulo)
    ax.set_xlabel(xlabel)
    fig.tight_layout()
    salvar_figura(fig, nome_figura)
    plt.show()

## Carregamento e verificação do JSON
A execução para se o arquivo não existir ou se falhar alguma verificação de integridade, de recorte ou de marcação (skill 02, seção 4.2, item 4).

In [ ]:
CAMINHO_JSON = PASTA / ARQUIVO_JSON
DOCUMENTO, VERIFICACOES = carregar_e_verificar(CAMINHO_JSON)
SHA256_JSON = sha256_arquivo(CAMINHO_JSON)
display(VERIFICACOES.set_index("verificação"))
if (VERIFICACOES["resultado"] != "ok").any():
    raise RuntimeError("O JSON não passou na verificação: a análise não continua (skill 02, seção 4.2, item 4).")

DOC, EXT = DOCUMENTO["document"], DOCUMENTO["extraction"]
print(f"Documento: {DOC['title']} — {ROTULO}")
print(f"Insumo: {ARQUIVO_JSON}  |  SHA-256 do JSON: {SHA256_JSON}")
recorte_txt = "documento inteiro" if EXT["cut"] is None else EXT["cut"].replace("_", " ")
print(f"Recorte: páginas {EXT['page_range'][0]} a {EXT['page_range'][1]} do PDF ({recorte_txt}); "
      f"{br(EXT['units_extracted'])} unidades; {br(EXT['words_extracted'])} palavras separadas por espaço (contagem da skill 01)")
print(f"Marcações: {br(EXT['marked_words'])} palavras com '{MARCADOR_REPETICAO}' em "
      f"{br(sum(p['occurrences'] for p in EXT['repetition_patterns']))} ocorrências de "
      f"{br(len(EXT['repetition_patterns']))} padrões do catálogo")
print(f"PDF de origem: {DOC['source_file']} (SHA-256 {DOC['source_sha256']}); extração de {EXT['date']}, "
      f"{EXT['skill']} versão {EXT['skill_version']}")

## Pré-processamento e perfil do documento
Filtros aplicados e efeito de cada um, unidades analisadas, totais de tokens antes e depois da retirada do excesso marcado e **lemas de classes de conteúdo descartados pela lista de *stopwords***, antes de qualquer visualização. Também ficam aqui as ênclises tratadas, os hífens usados como travessão, as correções de lema aplicadas e as sequências que não são tokens.

In [ ]:
nlp = spacy.load(MODELO_SPACY)
R = processar(DOCUMENTO, nlp)
T = R["itens"]
N_TOKENS = int(T["conta"].sum())               # denominador das taxas (sem o excesso marcado)
N_TOKENS_ANTES = int(T["palavra"].sum())       # antes da retirada do excesso marcado
N_EXCESSO = int((T["palavra"] & T["excesso"]).sum())
LEMAS = tabela_lemas(R)
CONT = T[T["conteudo"]]

# 1. Filtros de unidades e efeito de cada um
if R["filtros"]:
    filtros = pd.DataFrame([{"id": s["id"], "tipo": NOMES_TIPO.get(s["type"], s["type"]), "categoria": s["category"],
                             "palavras (regra de token)": len(RX_PALAVRA.findall(ler_marcacoes(s["text"])[0])), "motivo": m}
                            for s, m in R["filtros"]])
    display(filtros.groupby("motivo").agg(unidades=("id", "size"), palavras=("palavras (regra de token)", "sum")))
else:
    print("Filtros de unidades: nenhum. Títulos internos incluídos (INCLUIR_TITULOS_INTERNOS = True); nenhuma categoria "
          "nem unidade excluída; título da capa fora da contagem (INCLUIR_TITULO_NA_CONTAGEM = False).")

# 2. Unidades analisadas, por tipo e categoria
U = pd.DataFrame([{"id": s["id"], "tipo": s["type"], "categoria": s["category"]} for s in R["incluidas"]]).set_index("id")
U["tokens"] = T[T["conta"]].groupby("unidade").size().reindex(U.index, fill_value=0)
U["tokens_antes"] = T[T["palavra"]].groupby("unidade").size().reindex(U.index, fill_value=0)
print("\nUnidades analisadas, por tipo e categoria:")
display(pd.crosstab(U["tipo"].map(NOMES_TIPO), U["categoria"], margins=True, margins_name="Total").rename_axis(index="tipo", columns="categoria"))

# 3. Totais de tokens, antes e depois da retirada do excesso marcado
nao_conteudo = T[T["conta"] & ~T["conteudo"]]
totais = pd.Series({
    "Unidades analisadas": len(U),
    "Tokens no texto, antes da retirada do excesso marcado": N_TOKENS_ANTES,
    "Tokens retirados como excesso marcado": N_EXCESSO,
    "Tokens do documento (denominador das taxas)": N_TOKENS,
    "Tokens de conteúdo": len(CONT),
    "Lemas de conteúdo distintos": int(CONT["lema"].nunique()),
    "Tokens fora do conteúdo — classe gramatical (artigos, preposições, pronomes etc.)": int((~nao_conteudo["classe_conteudo"]).sum()),
    "Tokens fora do conteúdo — classe de conteúdo na lista de stopwords": int((nao_conteudo["classe_conteudo"] & nao_conteudo["stopword"] & ~nao_conteudo["modal"]).sum()),
    "Tokens fora do conteúdo — verbos modais (dever, poder)": int(nao_conteudo["modal"].sum()),
    "Tokens fora do conteúdo — lema com menos de 2 caracteres": int((nao_conteudo["classe_conteudo"] & ~nao_conteudo["stopword"] & ~nao_conteudo["modal"]).sum()),
}, name="valor")
print("\nTotais:")
display(tabela_br(totais.to_frame()))

# 4. Lemas de classes de conteúdo descartados pela lista de stopwords do spaCy (todos, com frequências)
SW = T[T["conta"] & T["classe_conteudo"] & T["stopword"] & ~T["modal"]]
DESCARTADOS = (SW.groupby("lema").agg(frequência=("lema", "size"), classes=("upos", lambda s: ", ".join(sorted(set(s)))))
               .sort_values(["frequência"], ascending=False).reset_index())
DESCARTADOS["por_mil"] = DESCARTADOS["frequência"] / N_TOKENS * 1000
limite_top = LEMAS.iloc[min(TOP_N_LEMAS, len(LEMAS)) - 1]["frequência"]
entrariam = DESCARTADOS[DESCARTADOS["frequência"] >= limite_top]
print(f"\nLemas de classes de conteúdo descartados pela lista de stopwords: {br(len(DESCARTADOS))} lemas, "
      f"{br(int(DESCARTADOS['frequência'].sum()))} ocorrências (continuam no denominador).")
if len(entrariam):
    print("⚠️ Com a frequência que têm, entrariam entre os " + str(TOP_N_LEMAS) + " unigramas se fossem mantidos: "
          + "; ".join(f"{r.lema} ({br(r.frequência)})" for r in entrariam.itertuples())
          + ". Manter algum deles (STOPWORDS_MANTIDAS) é decisão do pesquisador.")
with pd.option_context("display.max_rows", None):
    display(tabela_br(DESCARTADOS.set_index("lema"), {"por_mil": 2}))

# 5. Verbos modais (fora da contagem de conteúdo; medidos à parte)
mod = T[T["conta"] & T["modal"]].groupby("lema").size()
print("Verbos modais fora da contagem de conteúdo: "
      + ("; ".join(f"{l} = {br(n)} ({br(n / N_TOKENS * 1000, 2)} por mil)" for l, n in mod.items()) if len(mod) else "nenhum"))

# 6. Ênclises separadas (passo 2) e hífens usados como travessão
if R["especiais"]:
    esp = pd.DataFrame(R["especiais"], columns=["forma no texto", "tratamento", "verbo / partes", "lema do verbo",
                                                "classe (passagem auxiliar)", "pronome"])
    esp["lema final (após CORRECOES_DE_LEMA)"] = [CORRECOES_DE_LEMA.get(l, l) if l else "" for l in esp["lema do verbo"]]
    esp = esp.groupby(list(esp.columns)).size().rename("ocorrências").reset_index()
    print(f"\nFormas com hífen tratadas à parte: {br(int(esp['ocorrências'].sum()))} ocorrências.")
    with pd.option_context("display.max_rows", None):
        display(esp.set_index("forma no texto"))

# 7. Correções de lema aplicadas neste documento (CORRECOES_DE_LEMA)
corr = (T[T["corrigido"] & T["palavra"]].groupby(["lema_modelo", "lema", "upos"]).size().rename("ocorrências")
        .reset_index().rename(columns={"lema_modelo": "lema do modelo", "lema": "lema corrigido", "upos": "classe"})
        .sort_values(["ocorrências", "lema do modelo"], ascending=[False, True]))
print(f"\nCorreções de lema aplicadas: {br(int(corr['ocorrências'].sum()))} ocorrências, {br(len(corr))} pares (lema do modelo → lema corrigido).")
with pd.option_context("display.max_rows", None):
    display(corr.set_index("lema do modelo"))

# 8. Sequências com letras que não são tokens pela definição (números, símbolos, endereços eletrônicos)
outros = T[~T["palavra"] & T["texto"].str.contains(r"[^\W\d_]", regex=True)]
urls = outros["texto"].str.contains(r"://|www\.|@", regex=True)
print(f"\nNão são tokens (definição do passo 3): {br(int(urls.sum()))} endereços eletrônicos e "
      f"{br(int((~urls).sum()))} outras sequências com letras e algarismos ou símbolos: "
      + ", ".join(f"{x} ({br(n)})" for x, n in outros[~urls]["texto"].value_counts().items()))

#### Pontos do perfil para a decisão do pesquisador
- **Palavras de conteúdo retiradas pela lista de *stopwords*.** A lista tirou da contagem 94 lemas de classes de conteúdo, com 771 ocorrências, que continuam no denominador. Cinco deles entrariam entre os 30 unigramas se fossem mantidos: *mais* (41), *também* (40), *novo* (36), *sistema* (35) e *não* (33). *Sistema* é substantivo de conteúdo e *novo* tem carga semântica; *mais*, *também* e *não* são advérbios. A decisão (`STOPWORDS_MANTIDAS`) é do pesquisador.
- **Modais.** *Poder* (41; 2,64 por mil) e *dever* (21; 1,35 por mil).
- **Correções de lema.** Foram corrigidas 211 ocorrências, em 136 pares (tabela acima).
- **Ênclises.** Em 43 formas verbais com pronome (*estima-se*, *busca-se*), o verbo foi separado do pronome.
- **Não são tokens.** São 31 sequências com algarismos ou símbolos (*TOP500*, *GPT-3*, *R$*, *2023b*).
- ***Dados* e *dado*.** O modelo os reúne em *dado*. Separá-los é decisão do pesquisador.

## Repetições marcadas e retiradas
Catálogo dos padrões (expressão, circunstância, ocorrência mantida e ocorrências retiradas), tabela por termo e texto explicativo, com os números desta execução (skill 02, seção 3.2, item 5).

In [ ]:
CATALOGO = EXT["repetition_patterns"]
OC = pd.DataFrame(R["ocorrencias"])
if not CATALOGO:
    RETIRADAS_POR_LEMA = pd.DataFrame(columns=["lema", "retiradas"])
    display(Markdown(f"**Este JSON não tem padrões marcados**: o catálogo `repetition_patterns` está vazio e nenhuma palavra "
                     f"tem '{MARCADOR_REPETICAO}'. **Nada foi retirado por esta regra**: todas as ocorrências de todos os "
                     f"termos do {ROTULO} ficam na análise ({br(N_TOKENS)} tokens)."))
else:
    # 1. Catálogo: expressão, circunstância, ocorrência mantida e ocorrências retiradas
    linhas = []
    for p in CATALOGO:
        o = OC[OC["padrao"] == p["id"]]
        mantida = o[~o["excesso"]]
        linhas.append({"id": p["id"], "expressão": p["expression"], "marcada como": p["marked_as"],
                       "circunstância (where)": p["where"], "ocorrências": len(o),
                       "mantida: id da unidade": int(mantida["unidade"].iloc[0]) if len(mantida) else None,
                       "ocorrências retiradas": int(o["excesso"].sum()),
                       "palavras retiradas": int(sum(len(x) for x in o[o["excesso"]]["palavras"]))})
    CAT_TAB = pd.DataFrame(linhas).set_index("id")
    with pd.option_context("display.max_colwidth", None):
        display(tabela_br(CAT_TAB))
    # 2. Tabela por termo: lema, total no texto, retiradas (e de qual padrão), ficaram, taxa antes e depois
    marc = T[T["padrao"].notna() & T["palavra"]]
    ret = marc[marc["excesso"]]
    termos = []
    for lema, g in marc.groupby("lema"):
        conteudo = bool(g["conteudo_sem_retirada"].any())
        base = T[(T["lema"] == lema) & (T["conteudo_sem_retirada"] if conteudo else T["palavra"])]
        total = len(base)
        r = ret[ret["lema"] == lema]
        termos.append({"lema": lema, "na contagem de conteúdo": "sim" if conteudo else "não (palavra gramatical)",
                       "padrões": ", ".join(str(int(x)) for x in sorted(r["padrao"].unique())) if len(r) else "–",
                       "total no texto": total, "retiradas (excesso)": len(r), "ficaram": total - len(r),
                       "por mil antes": total / N_TOKENS_ANTES * 1000, "por mil depois": (total - len(r)) / N_TOKENS * 1000})
    TERMOS_RETIRADOS = pd.DataFrame(termos).sort_values(["retiradas (excesso)", "lema"], ascending=[False, True]).set_index("lema")
    display(tabela_br(TERMOS_RETIRADOS, {"por mil antes": 2, "por mil depois": 2}))
    RETIRADAS_POR_LEMA = TERMOS_RETIRADOS["retiradas (excesso)"].rename("retiradas").reset_index()
    # 3. Texto explicativo, com os números desta execução
    n_oc = int(OC["excesso"].sum())
    conteudo_ret = TERMOS_RETIRADOS[TERMOS_RETIRADOS["na contagem de conteúdo"] == "sim"]
    gram_ret = TERMOS_RETIRADOS[TERMOS_RETIRADOS["na contagem de conteúdo"] != "sim"]
    fora = []
    for lema, r in conteudo_ret.iterrows():
        corrido = int(((T["lema"] == lema) & T["conteudo_sem_retirada"] & T["padrao"].isna()).sum())
        fora.append(f"*{lema}* ({br(int(r['retiradas (excesso)']))} retiradas; ficaram {br(int(r['ficaram']))}, "
                    f"das quais {br(corrido)} no texto corrido, sem asterisco)")
    display(Markdown(
        f"**O que saiu e por quê.** No {ROTULO}, o catálogo da skill 01 traz {br(len(CATALOGO))} padrões de repetição, "
        f"todos rótulos fixos das fichas das ações dos anexos (a circunstância de cada um está na coluna *where*). "
        f"De cada padrão ficou a **primeira ocorrência** do documento (coluna *mantida*); as outras "
        f"**{br(n_oc)} ocorrências**, que são a repetição em excesso, saíram, com **{br(N_EXCESSO)} palavras**. "
        f"Elas saíram da contagem dos termos **e** do total de tokens: {br(N_TOKENS_ANTES)} tokens antes, "
        f"{br(N_TOKENS)} depois. Não formam bigramas nem entram no alcance ou na dispersão.\n\n"
        f"**Termos de conteúdo afetados:** " + "; ".join(fora) + ".\n\n"
        f"**Palavras gramaticais retiradas dos mesmos rótulos** (só saem do denominador): "
        + ", ".join(f"*{l}* ({br(int(r['retiradas (excesso)']))})" for l, r in gram_ret.iterrows()) + ".\n\n"
        f"**Nenhuma ocorrência sem asterisco foi retirada.** Nenhum termo desapareceu da análise por causa da marcação, "
        f"porque a primeira ocorrência de cada padrão ficou."))

## Possível inflação não marcada (mostrada, não retirada)
Repetições de texto **sem asterisco** que podem pesar nas contagens. Nada aqui é retirado: o notebook mostra e mede, e a decisão é do pesquisador. Uma marcação nova só pode ser feita pela skill 01 (skill 02, seções 3.6 e 4.1, passo 6).

In [ ]:
# Possível inflação NÃO marcada: o notebook mostra e mede; nada aqui é retirado (skill 02, seções 3.6 e 4.1, passo 6).
TEXTOS = pd.Series({s["id"]: R["limpos"][s["id"]] for s in R["incluidas"]})
TIPOS = pd.Series({s["id"]: s["type"] for s in R["incluidas"]})


def conteudo_alem_da_primeira(ids):
    """Tokens de conteúdo das unidades indicadas, sem contar a primeira delas."""
    g = T[T["unidade"].isin(ids) & T["conteudo"]]
    return len(g) - int((g["unidade"] == min(ids)).sum()), sorted(set(g["lema"]))


# 1. Unidades com texto idêntico (títulos e itens que se repetem literalmente)
linhas = []
for texto, n in TEXTOS.value_counts().items():
    if n < 2:
        continue
    ids = list(TEXTOS[TEXTOS == texto].index)
    extra, lemas = conteudo_alem_da_primeira(ids)
    linhas.append({"texto (literal)": texto, "tipo": NOMES_TIPO.get(TIPOS[ids[0]], TIPOS[ids[0]]), "unidades": n,
                   "tokens de conteúdo além da 1ª": extra, "lemas de conteúdo": ", ".join(lemas) or "–"})
REPETIDAS = pd.DataFrame(linhas)
print(f"1. Unidades de texto idêntico: {br(len(REPETIDAS))} textos repetidos, em "
      f"{br(int(REPETIDAS['unidades'].sum()) if len(REPETIDAS) else 0)} unidades.")
if len(REPETIDAS):
    with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
        display(tabela_br(REPETIDAS.sort_values(["unidades", "texto (literal)"], ascending=[False, True]).set_index("texto (literal)")))

# 2. Valores repetidos nos campos das fichas (texto que segue um rótulo marcado), quando houver fichas
linhas = []
valores = {}
for s in R["incluidas"]:
    if not s["marked_patterns"] or s["type"] == "heading":   # títulos das ações não são campo de ficha
        continue
    limpo, marcadas = ler_marcacoes(s["text"])
    grupos = agrupar_ocorrencias(limpo, marcadas)
    fim = grupos[0][-1][1]
    valor = re.sub(r"^[\s:]+", "", limpo[fim:]).strip()
    valores.setdefault((s["marked_patterns"][0], valor), []).append(s["id"])
for (padrao, valor), ids in valores.items():
    if len(ids) < 2:
        continue
    extra, lemas = conteudo_alem_da_primeira(ids)
    rotulo = next(p["expression"] for p in EXT["repetition_patterns"] if p["id"] == padrao)
    linhas.append({"campo (padrão)": rotulo, "valor (literal)": valor, "unidades": len(ids),
                   "tokens de conteúdo além da 1ª": extra, "lemas de conteúdo": ", ".join(lemas) or "–"})
VALORES_REPETIDOS = pd.DataFrame(linhas)
if EXT["repetition_patterns"]:
    print(f"\n2. Valores repetidos nos campos das fichas: {br(len(VALORES_REPETIDOS))} valores que aparecem em 2 ou mais fichas.")
    if len(VALORES_REPETIDOS):
        with pd.option_context("display.max_colwidth", None, "display.max_rows", None):
            display(tabela_br(VALORES_REPETIDOS.sort_values(["unidades", "valor (literal)"], ascending=[False, True])
                              .set_index(["campo (padrão)", "valor (literal)"])))
else:
    print("\n2. Valores repetidos nos campos das fichas: não se aplica (este JSON não tem fichas com rótulos marcados).")

# 3. Expressões de 4 palavras que se repetem em 5 ou mais unidades (fórmulas do texto)
quatro = defaultdict(set)
for uid, g in T[T["conta"]].groupby("unidade"):
    for sent, h in g.groupby("sent"):
        palavras = [x.lower() for x in h["texto"]]
        for i in range(len(palavras) - 3):
            quatro[" ".join(palavras[i:i + 4])].add(uid)
FORMULAS = (pd.DataFrame([{"expressão de 4 palavras": k, "unidades": len(v)} for k, v in quatro.items() if len(v) >= 5])
            .sort_values(["unidades", "expressão de 4 palavras"], ascending=[False, True]).head(25) if quatro else pd.DataFrame())
print(f"\n3. Expressões de 4 palavras em 5 ou mais unidades: {br(len(FORMULAS))} (até 25 exibidas).")
if len(FORMULAS):
    display(tabela_br(FORMULAS.set_index("expressão de 4 palavras")))

# 4. Palavras funcionais do inglês contadas como conteúdo (fora da lista de stopwords do português)
FUNCIONAIS_INGLES = ["a", "an", "and", "as", "at", "by", "for", "from", "in", "of", "on", "or", "the", "to", "with"]
ing = CONT[CONT["lema"].isin(FUNCIONAIS_INGLES) & CONT["texto"].str.match(r"^[A-Za-z]+$")]
ing = ing.assign(trecho=[" ".join(T.loc[max(0, i - 2):i + 2, "texto"]) for i in ing.index])
print(f"\n4. Palavras funcionais do inglês contadas como conteúdo: {br(len(ing))} ocorrências"
      + (": " + "; ".join(f"{l} ({br(n)})" for l, n in ing["lema"].value_counts().items()) if len(ing) else "."))
if len(ing):
    with pd.option_context("display.max_rows", None):
        display(ing[["lema", "upos", "unidade", "trecho"]].set_index("lema"))

#### Leitura desta seção
- **Subtítulos repetidos.** Os subtítulos fixos dos eixos se repetem: *Onde estamos?* (5 unidades idênticas), *O que vamos fazer* (4) e *Onde queremos chegar* (4), além de variantes com ou sem ponto de interrogação, que não contam como idênticas. Só *Onde queremos chegar* tem palavras de conteúdo (*querer*, *chegar*): 6 tokens além da primeira ocorrência. O bigrama *queremos chegar* (6) vem desses subtítulos e entrou entre os bigramas exibidos pelo empate.
- **Fórmulas do texto.**
  - *Este programa visa a* (12 unidades) abre as descrições dos programas e sustenta o bigrama *programa visa* (13) e parte do lema *visar* (40).
  - *IA para o bem de todos* aparece em 12 unidades.
  - *As ações propostas buscam* (7 unidades) sustenta *ações propostas* (11) e *propostas buscam* (7).
  - *Brasil se estabelecerá como* aparece em 5 unidades.
- **Inglês.** As palavras funcionais do inglês contam como conteúdo em 2 ocorrências (*and*).
- **Nada aqui foi retirado.** Marcar essas fórmulas (skill 01) ou excluí-las é decisão do pesquisador.

## Análises e visualizações
Todas as medidas são calculadas **sem o excesso marcado**. Cada visualização é seguida da síntese factual impressa pelo código e do bloco *Método e escolhas*, *Como ler*, *Interpretação* e *Fragilidades e ajustes possíveis*.

### Visualização 1 — Extensão das unidades textuais
*Visualização prévia: base inicial para a construção de novos gráficos.*

In [ ]:
ORDEM_TIPOS = [t for t in ("heading", "paragraph", "list_item") if t in set(U["tipo"])]
dados = [U.loc[U["tipo"] == t, "tokens"].to_numpy() for t in ORDEM_TIPOS]
fig, ax = plt.subplots(figsize=(8.6, 1.4 + 1.05 * len(ORDEM_TIPOS)))
bp = ax.boxplot(dados, orientation="horizontal", patch_artist=True, widths=0.5,
                medianprops={"color": COR_CONTRASTE, "linewidth": 1.8},
                flierprops={"marker": "o", "markersize": 3.5, "markerfacecolor": "none", "markeredgecolor": "0.35"})
for caixa in bp["boxes"]:
    caixa.set_facecolor(COR_DOCUMENTO)
    caixa.set_edgecolor(COR_CONTORNO)
ax.set_yticks(range(1, len(ORDEM_TIPOS) + 1))
ax.set_yticklabels([f"{NOMES_TIPO[t]}\n(n = {br(len(d))})" for t, d in zip(ORDEM_TIPOS, dados)])
ax.set_xlim(left=0)
ax.xaxis.set_major_formatter(EIXO_BR)
for i, d in enumerate(dados, start=1):
    ax.text(float(np.median(d)), i + 0.33, f"mediana {br(float(np.median(d)), 1)}", ha="center", va="bottom", fontsize=9)
ax.set_xlabel("tokens por unidade (palavras do texto, sem o excesso marcado)")
ax.set_title(f"Extensão das unidades textuais — {ROTULO}")
fig.tight_layout()
salvar_figura(fig, "extensao_unidades")
plt.show()

EXT_TIPO = U.groupby("tipo")["tokens"].agg(unidades="size", tokens="sum", média="mean", mediana="median", mínimo="min", máximo="max")
EXT_TIPO.index = EXT_TIPO.index.map(NOMES_TIPO)
EXT_CAT = U.groupby("categoria")["tokens"].agg(unidades="size", tokens="sum", média="mean", mediana="median", mínimo="min", máximo="max")
display(tabela_br(EXT_TIPO, {"média": 1, "mediana": 1}))
display(tabela_br(EXT_CAT, {"média": 1, "mediana": 1}))
salvar_tabela(U.reset_index(), "extensao_unidades_tabela")

maior = U["tokens"].idxmax()
afetadas = U[U["tokens"] < U["tokens_antes"]]
print("Síntese factual:")
print(f"- {br(len(U))} unidades e {br(N_TOKENS)} tokens; média de {br(U['tokens'].mean(), 1)} e mediana de "
      f"{br(U['tokens'].median(), 1)} tokens por unidade.")
for t, r in EXT_TIPO.iterrows():
    print(f"- {t}: {br(int(r['unidades']))} unidades, {br(int(r['tokens']))} tokens ({br_pct(r['tokens'] / N_TOKENS)} do total); "
          f"mediana {br(r['mediana'], 1)}, máximo {br(int(r['máximo']))}.")
print(f"- Unidade mais longa: id {maior} ({NOMES_TIPO[U.loc[maior, 'tipo']]}, categoria {U.loc[maior, 'categoria']}), "
      f"{br(int(U.loc[maior, 'tokens']))} tokens.")
print(f"- Retirada do excesso marcado: {br(len(afetadas))} unidades perderam tokens "
      f"({br(int((afetadas['tokens_antes'] - afetadas['tokens']).sum()))} no total)." if len(afetadas)
      else "- Retirada do excesso marcado: nenhuma unidade perdeu tokens (não há padrões marcados neste JSON).")

#### Método e escolhas
Número de tokens de cada uma das 284 unidades (palavras do texto, pela definição do passo 3 da skill), por tipo de unidade. Não houve retirada de excesso, porque o recorte sem anexos não tem padrões marcados. No diagrama de caixa, a caixa vai do 1º ao 3º quartil, o traço interno é a mediana, as hastes vão até 1,5 vez o intervalo interquartil e os círculos são as unidades além desse limite. As tabelas trazem as medidas por tipo e por categoria (`Foreword`, `Body`).

#### Como ler
Cada linha é um tipo de unidade, com o número de unidades entre parênteses; quanto mais à direita a caixa, mais longas as unidades desse tipo.

#### Interpretação
**Observação.**
- Os parágrafos (147) concentram 76,8% dos 15.559 tokens, com mediana de 73.
- Os itens de lista (83) somam 20,7%, com mediana de 29 e unidades de até 178 tokens.
- Os títulos internos (54) somam 2,5%, com mediana de 5,5 e títulos de até 37 tokens.
- Por categoria:
  - o corpo tem 272 unidades e 15.007 tokens;
  - a Apresentação, 12 unidades e 552 tokens.
- A unidade mais longa é um parágrafo (id 58, 217 tokens).

**Inferência.** O texto é sobretudo expositivo, em parágrafos. Os títulos internos são longos para títulos (mediana de 5,5 tokens), o que sugere títulos descritivos.

#### Fragilidades e ajustes possíveis
- A extensão depende da segmentação da extração: os parágrafos que atravessam a página foram unidos.
- Depende também da definição de token: números e siglas com algarismos, como *TOP500*, não contam.

### Visualização 2 — Os 30 principais unigramas (lemas de conteúdo)
*Visualização construída por comando do pesquisador (2026-10-13): os 30 principais unigramas, com o recorte ampliado até o desempate quando houver empate no ponto de corte.*

In [ ]:
TOP_LEMAS, INFO_LEMAS = recorte(LEMAS, TOP_N_LEMAS, ampliar_empates=AMPLIAR_EMPATES_LEMAS, chave="lema")
titulo = f"Os {TOP_N_LEMAS} unigramas (lemas de conteúdo) mais frequentes — {ROTULO}"
if INFO_LEMAS["exibidos"] > TOP_N_LEMAS:
    titulo += f"\n{INFO_LEMAS['exibidos']} exibidos: recorte ampliado até o desempate"
barras_horizontais(TOP_LEMAS, "lema", "frequência", titulo, "frequência absoluta (entre parênteses: por 1.000 tokens)",
                   INFO_LEMAS, "unigramas_mais_frequentes")

TAB_LEMAS = TOP_LEMAS[["lema", "frequência", "por_mil", "alcance_unidades", "alcance_pct", "DP_norm",
                       "retiradas_excesso", "frequência_antes"]].copy()
TAB_LEMAS["alcance_pct"] = 100 * TAB_LEMAS["alcance_pct"]
TAB_LEMAS.index = pd.RangeIndex(1, len(TAB_LEMAS) + 1, name="posição")
TAB_LEMAS = TAB_LEMAS.rename(columns={"por_mil": "por 1.000 tokens", "alcance_unidades": "alcance (unidades)",
                                      "alcance_pct": "alcance (%)", "DP_norm": "DP normalizado",
                                      "retiradas_excesso": "retiradas (excesso marcado)",
                                      "frequência_antes": "frequência antes da retirada"})
display(tabela_br(TAB_LEMAS, {"por 1.000 tokens": 2, "alcance (%)": 1, "DP normalizado": 2}))
salvar_tabela(LEMAS, "lemas_tabela_completa")

soma = int(TOP_LEMAS["frequência"].sum())
afet = TOP_LEMAS[TOP_LEMAS["retiradas_excesso"] > 0]
print("Síntese factual:")
print(f"- Recorte pedido: {TOP_N_LEMAS}; exibidos: {INFO_LEMAS['exibidos']}"
      + (f" (acrescentados por empate na frequência {br(int(INFO_LEMAS['limite']))}: {', '.join(INFO_LEMAS['acrescentados'])})"
         if INFO_LEMAS["acrescentados"] else " (sem empate no ponto de corte)")
      + (f"; primeiro lema fora: {INFO_LEMAS['primeiro_fora'][0]} ({br(int(INFO_LEMAS['primeiro_fora'][1]))})."
         if INFO_LEMAS["primeiro_fora"] else "."))
print(f"- {br(len(LEMAS))} lemas de conteúdo distintos e {br(len(CONT))} tokens de conteúdo; os {INFO_LEMAS['exibidos']} exibidos "
      f"somam {br(soma)} ocorrências ({br_pct(soma / len(CONT))} dos tokens de conteúdo).")
print("- Mais frequentes: " + "; ".join(f"{r.lema} = {br(r.frequência)} ({br(r.por_mil, 2)} por mil, em "
                                         f"{br_pct(r.alcance_pct)} das unidades)" for r in TOP_LEMAS.head(5).itertuples()) + ".")
print("- Lista exibida (posição. lema: frequência): "
      + "; ".join(f"{i}. {r.lema}: {br(r.frequência)}" for i, r in enumerate(TOP_LEMAS.itertuples(), start=1)) + ".")
print("- Retirada do excesso marcado nos lemas exibidos: "
      + ("; ".join(f"{r.lema}: {br(r.frequência_antes)} → {br(r.frequência)} (−{br(r.retiradas_excesso)})" for r in afet.itertuples())
         if len(afet) else "nenhum lema exibido teve ocorrências retiradas") + ".")

#### Método e escolhas
- **Medida.** Frequência absoluta dos lemas de conteúdo. A taxa por 1.000 tokens usa o denominador de 15.559, e o alcance é medido em 284 unidades.
- **Excesso marcado.** Não houve retirada: o recorte não tem padrões marcados.
- **Recorte (comando do pesquisador).** 30 unigramas, ampliado no empate. O 30º lugar tem frequência 33, a mesma de *aplicação* (29º) e *estratégico* (30º), e por isso *programa* (33) entrou como 31º. O primeiro fora é *estabelecer* (32).

#### Como ler
Barra = frequência absoluta; entre parênteses, a frequência por 1.000 tokens. A linha tracejada marca o fim dos 30 pedidos; abaixo dela está o lema empatado.

#### Interpretação
**Observação.**
- O lema *ia* tem 393 ocorrências (25,26 por mil) e aparece em 65,1% das unidades, muito à frente de *desenvolvimento* (141).
- Seguem os termos ligados ao país: *brasil* (111), *país* (83), *brasileiro* (75), *nacional* (68).
- Vêm depois:
  - desenvolvimento, inovação e capacidade técnica: *tecnologia* (65), *inovação* (64), *tecnológico* (64), *solução* (50), *infraestrutura* (49), *modelo* (37), *capacidade* (34);
  - verbos de propósito: *promover* (44), *visar* (40), *garantir* (38);
  - vocabulário de plano: *ação* (73), *desafio* (48), *potencial* (44), *impacto* (43).
- Os 31 lemas somam 2.115 ocorrências, 25,6% dos 8.247 tokens de conteúdo.

**Inferência.** O vocabulário mais frequente combina a afirmação nacional com o desenvolvimento de capacidades tecnológicas e de infraestrutura, em linguagem de plano (ações, desafios, impactos, objetivos a promover e garantir).

#### Fragilidades e ajustes possíveis
- *Visar* (40) é reforçado pela fórmula *Este programa visa a*, presente em 12 unidades (ver *Possível inflação não marcada*).
- *Ação* reúne usos diferentes (*ações de impacto*, *ações estruturantes*, *ações propostas*).
- *Sistema* (35) ficou fora pela lista de *stopwords*.
- *Dado* reúne *dados* e *dado*.
- *Ia* e *inteligência artificial* não foram unificados.

### Visualização 3 — Distribuição dos lemas ao longo do documento
*Visualização prévia: base inicial para a construção de novos gráficos.*

In [ ]:
TOP_DISP, INFO_DISP = recorte(LEMAS, TOP_N_DISPERSAO, ampliar_empates=False, chave="lema")
ORDEM_DISP = list(TOP_DISP["lema"])
POS = T[T["conteudo"] & T["lema"].isin(ORDEM_DISP)][["lema", "posicao"]]
fig, ax = plt.subplots(figsize=(9.8, 0.42 * len(ORDEM_DISP) + 1.9))
# faixas das partes que não são o corpo (Apresentação e Anexos), para situar a leitura
LIM = T[T["conta"]].groupby("unidade")["posicao"].agg(["min", "max"])
faixas, atual = [], None
for uid, cat in U["categoria"].items():
    if uid not in LIM.index:
        continue
    if cat in ("Foreword", "Annex"):
        if atual and atual[0] == cat:
            atual[2] = LIM.loc[uid, "max"]
        else:
            atual = [cat, LIM.loc[uid, "min"], LIM.loc[uid, "max"]]
            faixas.append(atual)
    elif cat != "Box":
        atual = None
for cat, x0, x1 in faixas:
    ax.axvspan(x0 - 0.5, x1 + 0.5, color="0.92", zorder=0)
    ax.text((x0 + x1) / 2, -0.9, {"Foreword": "Apresentação", "Annex": "Anexos"}[cat], ha="center", va="bottom", fontsize=9, color="0.3")
for i, lema in enumerate(ORDEM_DISP):
    xs = POS.loc[POS["lema"] == lema, "posicao"]
    ax.vlines(xs, i - 0.36, i + 0.36, color=COR_CONTORNO, linewidth=1.3)   # contorno escuro
    ax.vlines(xs, i - 0.33, i + 0.33, color=COR_DOCUMENTO, linewidth=0.7)
ax.set_yticks(range(len(ORDEM_DISP)))
ax.set_yticklabels(ORDEM_DISP)
ax.set_ylim(len(ORDEM_DISP) - 0.4, -1.2)
ax.set_xlim(0, N_TOKENS)
ax.xaxis.set_major_formatter(EIXO_BR)
for i, r in enumerate(TOP_DISP.itertuples()):
    ax.text(N_TOKENS * 1.01, i, f"DP = {br(r.DP_norm, 2)}", va="center", fontsize=9)
ax.set_xlabel("posição no documento (tokens na ordem do texto, sem o excesso marcado)")
ax.set_title(f"Distribuição dos {len(ORDEM_DISP)} lemas de conteúdo mais frequentes ao longo do texto — {ROTULO}")
fig.tight_layout()
salvar_figura(fig, "dispersao_lemas")
plt.show()

TAB_DISP = TOP_DISP[["lema", "frequência", "alcance_unidades", "alcance_pct", "DP_norm"]].copy()
TAB_DISP["alcance_pct"] = 100 * TAB_DISP["alcance_pct"]
TAB_DISP.index = pd.RangeIndex(1, len(TAB_DISP) + 1, name="posição")
TAB_DISP = TAB_DISP.rename(columns={"alcance_unidades": "alcance (unidades)", "alcance_pct": "alcance (%)", "DP_norm": "DP normalizado"})
display(tabela_br(TAB_DISP, {"alcance (%)": 1, "DP normalizado": 2}))

ordenado = TOP_DISP.sort_values("DP_norm")
print("Síntese factual:")
print(f"- {len(ORDEM_DISP)} lemas mais frequentes (desempate alfabético"
      + (f"; empatados que ficaram de fora: {', '.join(INFO_DISP['fora_empatados'])}" if INFO_DISP["fora_empatados"] else "; sem empate no corte")
      + f"). Partes: {br(len(U))} unidades.")
print(f"- Mais bem distribuídos (DP menor): " + "; ".join(f"{r.lema} ({br(r.DP_norm, 2)})" for r in ordenado.head(3).itertuples()) + ".")
print(f"- Mais concentrados (DP maior): " + "; ".join(f"{r.lema} ({br(r.DP_norm, 2)})" for r in ordenado.tail(3).iloc[::-1].itertuples()) + ".")
print(f"- Faixas cinza: " + ("; ".join(f"{ {'Foreword': 'Apresentação', 'Annex': 'Anexos'}[c]} (tokens {br(int(a))} a {br(int(b))})" for c, a, b in faixas)
                             if faixas else "nenhuma") + ".")

#### Método e escolhas
- **Lemas.** Os 15 lemas de conteúdo mais frequentes, pelo padrão da skill (desempate alfabético; não houve empate no corte).
- **Traços.** Cada traço é uma ocorrência, na posição do token no documento (1 a 15.559).
- **DP.** O DP de Gries, normalizado por Lijffijt e Gries, é calculado com as 284 unidades como partes: 0 = proporcional ao tamanho; 1 = concentração máxima.
- **Faixa cinza.** Marca a Apresentação (tokens 1 a 552).
- **Excesso marcado.** Sem retirada.

#### Como ler
Traços espalhados por todo o eixo indicam uso ao longo do documento; aglomerados indicam concentração em trechos. O DP está à direita de cada linha.

#### Interpretação
**Observação.**
- O lema mais bem distribuído é *ia* (DP 0,32), seguido por *desenvolvimento* (0,49), *brasil* (0,62) e *país* (0,67).
- Os mais concentrados são *ação* (0,81), *inteligência* (0,78) e *artificial* (0,77).
- Na figura:
  - *inteligência* e *artificial* se adensam até cerca do token 3.000 e ficam esparsos depois;
  - *ia* se torna denso e contínuo a partir do token 3.800, aproximadamente;
  - *ação* forma um bloco entre os tokens 8.300 e 9.000;
  - *público* se adensa entre 12.200 e 13.300.

**Inferência.**
- Depois das partes iniciais, o documento passa a usar sobretudo a sigla *IA*.
- *Ação* e *público* se concentram em seções específicas, que só o agrupamento por `path` permite identificar.

#### Fragilidades e ajustes possíveis
- O DP usa as unidades do JSON como partes. Num documento com muitas unidades curtas (títulos e itens), lemas de frequência moderada tendem a ter DP alto mesmo quando aparecem em várias seções.
- O DP diz quanto o lema está concentrado, mas não onde; por isso a figura acompanha o valor.
- As posições citadas na interpretação foram lidas na figura e são aproximadas.

### Visualização 4 — Os 25 principais bigramas
*Visualização construída por comando do pesquisador (2026-10-13): os 25 principais bigramas, com o recorte ampliado até o desempate quando houver empate no ponto de corte.*

In [ ]:
BIG = tabela_bigramas(R)
BIG_VALIDOS = BIG[BIG["frequência"] >= FREQ_MINIMA_BIGRAMA].reset_index(drop=True)
TOP_BIG, INFO_BIG = recorte(BIG_VALIDOS, TOP_N_BIGRAMAS, ampliar_empates=AMPLIAR_EMPATES_BIGRAMAS, chave="bigrama")
titulo = f"Os {TOP_N_BIGRAMAS} bigramas mais frequentes — {ROTULO}"
if INFO_BIG["exibidos"] > TOP_N_BIGRAMAS:
    titulo += f"\n{INFO_BIG['exibidos']} exibidos: recorte ampliado até o desempate"
barras_horizontais(TOP_BIG, "bigrama", "frequência", titulo, "frequência absoluta (entre parênteses: por 1.000 tokens)",
                   INFO_BIG, "bigramas_mais_frequentes")

TAB_BIG = TOP_BIG[["bigrama", "lemas", "frequência", "por_mil", "alcance_unidades", "retiradas_excesso", "formas_no_texto",
                   "lemas_reunidos"]].copy()
TAB_BIG.index = pd.RangeIndex(1, len(TAB_BIG) + 1, name="posição")
TAB_BIG = TAB_BIG.rename(columns={"bigrama": "bigrama (forma mais frequente no texto)", "lemas": "par de lemas",
                                  "por_mil": "por 1.000 tokens", "alcance_unidades": "alcance (unidades)",
                                  "retiradas_excesso": "retiradas (excesso marcado)", "formas_no_texto": "formas no texto",
                                  "lemas_reunidos": "outros pares de lemas com a mesma forma escrita (reunidos)"})
with pd.option_context("display.max_colwidth", None):
    display(tabela_br(TAB_BIG, {"por 1.000 tokens": 2}))
salvar_tabela(BIG, "bigramas_tabela_completa")

repetidas_rotulo = BIG[BIG["bigrama"].duplicated(keep=False)]
afet = TOP_BIG[TOP_BIG["retiradas_excesso"] > 0]
print("Síntese factual:")
print(f"- {br(len(BIG))} bigramas distintos; {br(len(BIG_VALIDOS))} com frequência ≥ {FREQ_MINIMA_BIGRAMA}.")
print(f"- Recorte pedido: {TOP_N_BIGRAMAS}; exibidos: {INFO_BIG['exibidos']}"
      + (f" (acrescentados por empate na frequência {br(int(INFO_BIG['limite']))}: {', '.join(INFO_BIG['acrescentados'])})"
         if INFO_BIG["acrescentados"] else " (sem empate no ponto de corte)")
      + (f"; primeiro bigrama fora: {INFO_BIG['primeiro_fora'][0]} ({br(int(INFO_BIG['primeiro_fora'][1]))})."
         if INFO_BIG["primeiro_fora"] else "."))
print("- Mais frequentes: " + "; ".join(f"{r.bigrama} = {br(r.frequência)} ({br(r.por_mil, 2)} por mil)"
                                         for r in TOP_BIG.head(5).itertuples()) + ".")
print("- Lista exibida (posição. bigrama: frequência): "
      + "; ".join(f"{i}. {r.bigrama}: {br(r.frequência)}" for i, r in enumerate(TOP_BIG.itertuples(), start=1)) + ".")
reun = TOP_BIG[TOP_BIG["lemas_reunidos"] != "–"]
print("- Pares de lemas reunidos por terem a mesma forma escrita (nos exibidos): "
      + ("; ".join(f"{r.bigrama} ← {r.lemas_reunidos}" for r in reun.itertuples()) if len(reun) else "nenhum") + ".")
print("- Retirada do excesso marcado nos bigramas exibidos: "
      + ("; ".join(f"{r.bigrama}: {br(r.frequência_antes)} → {br(r.frequência)}" for r in afet.itertuples())
         if len(afet) else "nenhum bigrama exibido perdeu ocorrências") + ".")
if len(repetidas_rotulo):
    print("- Atenção: a mesma forma escrita aparece em mais de um par de lemas: "
          + "; ".join(f"{b} ({', '.join(g['lemas'])})" for b, g in repetidas_rotulo.groupby("bigrama")) + ".")

#### Método e escolhas
- **Definição.** Pares de lemas de conteúdo adjacentes no texto, na mesma frase, sem pontuação nem palavra retirada entre eles; por isso, expressões ligadas por preposição (*proteção de dados*, *soluções de IA*) não formam bigrama. Frequência mínima: 2.
- **Recorte (comando do pesquisador).** 25 bigramas, ampliado no empate. A frequência do 25º é 6, e entraram mais 10 empatados, num total de 35. O primeiro fora é *autonomia tecnológica* (5).
- **Formas reunidas.** Ocorrências com a mesma forma escrita foram reunidas: *ações propostas* (1 ocorrência lida como *ação + propostas*) e *vantagem competitiva* (2 lidas como *vantagem + competitiva*).
- **Excesso marcado.** Sem retirada.

#### Como ler
O rótulo é a forma escrita mais frequente. A linha tracejada marca o fim dos 25 pedidos; abaixo dela estão os empatados.

#### Interpretação
**Observação.**
- *Inteligência artificial* lidera, com 51 ocorrências (3,28 por mil).
- Seguem *setor público* (18), *serviços públicos* (15), *programa visa* (13), *ações propostas* (11) e *matriz energética* (10).
- As demais expressões se agrupam assim:
  - energia e sustentabilidade: *matriz energética*, *desenvolvimento sustentável*, *ia sustentável*, *eficiência energética*;
  - capacidade computacional: *alto desempenho*, *data centers*, *infraestrutura computacional*, *ia generativa*;
  - economia: *cadeia produtiva*, *inovação empresarial*, *soluções inovadoras*, *vantagem competitiva*;
  - Estado e sociedade: *setor público*, *serviços públicos*, *administração pública*, *governo federal*, *políticas públicas*, *sociedade brasileira*, *sociedade civil*, *população brasileira*;
  - posição do país: *líder global*, *contexto brasileiro*.

**Inferência.** As expressões recorrentes associam o plano a energia, capacidade computacional e economia, ao lado da atuação estatal.

#### Fragilidades e ajustes possíveis
- *Programa visa* (13), *ações propostas* (11) e *propostas buscam* (7) vêm de fórmulas repetidas na apresentação de programas e ações.
- *Queremos chegar* (6) vem dos subtítulos fixos dos eixos (ver *Possível inflação não marcada*).
- *Data centers* é expressão em inglês.
- Bigramas com preposição (*soluções de IA*) ficam fora por definição.
- Com o corte em 6, entraram dez empatados.

## Registro de execução
Parâmetros, versões do ambiente e do modelo, *hash* do insumo e lista de saídas gravadas.

In [ ]:
REGISTRO = pd.Series({
    "executado em": datetime.now().isoformat(timespec="seconds"),
    "notebook": f"{NOME_SAIDA}.ipynb",
    "insumo": ARQUIVO_JSON,
    "SHA-256 do insumo": SHA256_JSON,
    "PDF de origem (SHA-256)": f"{DOC['source_file']} ({DOC['source_sha256']})",
    "Python": platform.python_version(),
    "spaCy": spacy.__version__,
    "modelo spaCy": f"{nlp.meta['lang']}_{nlp.meta['name']} {nlp.meta['version']}",
    "pandas / numpy / matplotlib": f"{pd.__version__} / {np.__version__} / {matplotlib.__version__}",
    "RETIRAR_EXCESSO_MARCADO": RETIRAR_EXCESSO_MARCADO,
    "INCLUIR_TITULO_NA_CONTAGEM / INCLUIR_TITULOS_INTERNOS": f"{INCLUIR_TITULO_NA_CONTAGEM} / {INCLUIR_TITULOS_INTERNOS}",
    "CATEGORIAS_EXCLUIDAS / UNIDADES_EXCLUIDAS": f"{CATEGORIAS_EXCLUIDAS} / {UNIDADES_EXCLUIDAS}",
    "CLASSES_DE_CONTEUDO": ", ".join(CLASSES_DE_CONTEUDO),
    "TAMANHO_MINIMO_LEMA": TAMANHO_MINIMO_LEMA,
    "stopwords (spaCy + adicionais − mantidas)": f"{len(STOPWORDS_SPACY)} + {len(STOPWORDS_ADICIONAIS)} − {len(STOPWORDS_MANTIDAS)} = {len(STOPWORDS)}",
    "LEMAS_MODAIS": ", ".join(LEMAS_MODAIS),
    "PRONOMES_CLITICOS": ", ".join(PRONOMES_CLITICOS),
    "HIFEN_COMO_TRAVESSAO": ", ".join(HIFEN_COMO_TRAVESSAO),
    "CORRECOES_DE_LEMA": f"{len(CORRECOES_DE_LEMA)} entradas (aplicadas aqui: {br(int(T['corrigido'].sum()))} ocorrências)",
    "TOP_N_LEMAS / AMPLIAR_EMPATES_LEMAS": f"{TOP_N_LEMAS} / {AMPLIAR_EMPATES_LEMAS}",
    "TOP_N_BIGRAMAS / AMPLIAR_EMPATES_BIGRAMAS / FREQ_MINIMA_BIGRAMA": f"{TOP_N_BIGRAMAS} / {AMPLIAR_EMPATES_BIGRAMAS} / {FREQ_MINIMA_BIGRAMA}",
    "TOP_N_DISPERSAO": TOP_N_DISPERSAO,
    "COR_DOCUMENTO": COR_DOCUMENTO,
    "tokens (antes → depois da retirada do excesso)": f"{br(N_TOKENS_ANTES)} → {br(N_TOKENS)}",
    "saídas gravadas": ", ".join(SAIDAS_GRAVADAS) if SAIDAS_GRAVADAS else "nenhuma (SALVAR_SAIDAS = False)",
}, name="valor")
with pd.option_context("display.max_colwidth", None):
    display(REGISTRO.to_frame())

## Síntese
- **Insumo.** `PBIA_sem_anexos_extracao.json`, páginas 11 a 48: 284 unidades, 15.559 tokens, 8.247 tokens de conteúdo e 1.706 lemas distintos. Não há padrões marcados, e nada foi retirado como excesso.
- **Os 31 unigramas** (comando do pesquisador: 30, ampliado no empate): 1. *ia* (393); 2. *desenvolvimento* (141); 3. *brasil* (111); 4. *público* (88); 5. *país* (83); 6. *brasileiro* (75); 7. *ação* (73); 8. *dado* (72); 9. *nacional* (68); 10. *setor* (65); 11. *tecnologia* (65); 12. *inovação* (64); 13. *tecnológico* (64); 14. *artificial* (55); 15. *inteligência* (52); 16. *solução* (50); 17. *infraestrutura* (49); 18. *desafio* (48); 19. *uso* (48); 20. *potencial* (44); 21. *promover* (44); 22. *impacto* (43); 23. *visar* (40); 24. *garantir* (38); 25. *modelo* (37); 26. *pesquisa* (37); 27. *processo* (35); 28. *capacidade* (34); 29. *aplicação* (33); 30. *estratégico* (33); 31. *programa* (33.).
- **Os 35 bigramas** (comando do pesquisador: 25, ampliado no empate): 1. *inteligência artificial* (51); 2. *setor público* (18); 3. *serviços públicos* (15); 4. *programa visa* (13); 5. *ações propostas* (11); 6. *matriz energética* (10); 7. *ações estruturantes* (9); 8. *desenvolvimento sustentável* (9); 9. *cadeia produtiva* (8); 10. *plano brasileiro* (8); 11. *população brasileira* (8); 12. *potencial transformador* (8); 13. *setor privado* (8); 14. *administração pública* (7); 15. *governo federal* (7); 16. *ia sustentável* (7); 17. *inovação empresarial* (7); 18. *propostas buscam* (7); 19. *sociedade brasileira* (7); 20. *soluções inovadoras* (7); 21. *uso ético* (7); 22. *vantagem competitiva* (7); 23. *alto desempenho* (6); 24. *contexto brasileiro* (6); 25. *data centers* (6); 26. *desenvolvimento tecnológico* (6); 27. *eficiência energética* (6); 28. *ia generativa* (6); 29. *infraestrutura computacional* (6); 30. *longo prazo* (6); 31. *líder global* (6); 32. *políticas públicas* (6); 33. *queremos chegar* (6); 34. *sociedade civil* (6); 35. *transformação digital* (6.).
- **Pontos que dependem do pesquisador:**
  - manter *sistema* (35) e os demais lemas tirados pela lista de *stopwords*;
  - tratar as fórmulas *este programa visa a*, *as ações propostas buscam* e os subtítulos fixos dos eixos;
  - separar *dados* de *dado*.

## Seções acrescentadas por comando do pesquisador (2026-10-13)
As seções acima ficam intactas. As visualizações abaixo usam as tabelas completas já calculadas no notebook (`LEMAS` e `BIG_VALIDOS`): lemas de conteúdo e bigramas sem o excesso marcado, com o mesmo pré-processamento, as mesmas correções de lema e as mesmas regras de contagem. Os parâmetros de cada recorte estão na própria célula.

### Visualização 5 — Os 50 principais unigramas
*Visualização construída por comando do pesquisador (2026-10-13): os 50 principais unigramas, com o recorte ampliado até o desempate quando houver empate no ponto de corte.*

In [ ]:
# Parâmetros desta seção (comando do pesquisador, 2026-10-13): os 50 principais unigramas, ampliado até o desempate
N_UNIGRAMAS_50 = 50
AMPLIAR_EMPATES_UNIGRAMAS_50 = True

TOP_UNIGRAMAS_50, INFO_UNIGRAMAS_50 = recorte(LEMAS, N_UNIGRAMAS_50, ampliar_empates=AMPLIAR_EMPATES_UNIGRAMAS_50, chave="lema")
titulo = f"Os {N_UNIGRAMAS_50} principais unigramas (lemas de conteúdo) — {ROTULO}"
if INFO_UNIGRAMAS_50["exibidos"] > N_UNIGRAMAS_50:
    titulo += f"\n{INFO_UNIGRAMAS_50['exibidos']} exibidos: recorte ampliado até o desempate"
barras_horizontais(TOP_UNIGRAMAS_50, "lema", "frequência", titulo, "frequência absoluta (entre parênteses: por 1.000 tokens)",
                   INFO_UNIGRAMAS_50, "unigramas_top50")

TAB_UNIGRAMAS_50 = TOP_UNIGRAMAS_50[["lema", "frequência", "por_mil", "alcance_unidades", "alcance_pct", "DP_norm",
                                     "retiradas_excesso"]].copy()
TAB_UNIGRAMAS_50["alcance_pct"] = 100 * TAB_UNIGRAMAS_50["alcance_pct"]
TAB_UNIGRAMAS_50.index = pd.RangeIndex(1, len(TAB_UNIGRAMAS_50) + 1, name="posição")
TAB_UNIGRAMAS_50 = TAB_UNIGRAMAS_50.rename(columns={"por_mil": "por 1.000 tokens", "alcance_unidades": "alcance (unidades)",
                                                     "alcance_pct": "alcance (%)", "DP_norm": "DP normalizado",
                                                     "retiradas_excesso": "retiradas (excesso marcado)"})
with pd.option_context("display.max_rows", None):
    display(tabela_br(TAB_UNIGRAMAS_50, {"por 1.000 tokens": 2, "alcance (%)": 1, "DP normalizado": 2}))

info = INFO_UNIGRAMAS_50
soma = int(TOP_UNIGRAMAS_50["frequência"].sum())
print("Síntese factual:")
print(f"- Recorte pedido: {N_UNIGRAMAS_50}; exibidos: {info['exibidos']}"
      + (f" (acrescentados por empate na frequência {br(int(info['limite']))}: {', '.join(info['acrescentados'])})"
         if info["acrescentados"] else " (sem empate no ponto de corte)")
      + (f"; primeiro lema fora: {info['primeiro_fora'][0]} ({br(int(info['primeiro_fora'][1]))})." if info["primeiro_fora"] else "."))
print(f"- Os {info['exibidos']} lemas somam {br(soma)} ocorrências ({br_pct(soma / len(CONT))} dos {br(len(CONT))} tokens de conteúdo); "
      f"frequência entre {br(int(TOP_UNIGRAMAS_50['frequência'].min()))} e {br(int(TOP_UNIGRAMAS_50['frequência'].max()))}.")
print("- Lista exibida (posição. lema: frequência): "
      + "; ".join(f"{i}. {r.lema}: {br(r.frequência)}" for i, r in enumerate(TOP_UNIGRAMAS_50.itertuples(), start=1)) + ".")
print("- Retirada do excesso marcado: " + (f"{br(int(TOP_UNIGRAMAS_50['retiradas_excesso'].sum()))} ocorrências retiradas nos lemas exibidos."
                                           if TOP_UNIGRAMAS_50["retiradas_excesso"].sum() else "nenhuma (este JSON não tem padrões marcados)."))


#### Método e escolhas
- **Medida.** As mesmas da Visualização 2: lemas de conteúdo, taxa por 1.000 tokens com denominador de 15.559 e alcance em 284 unidades. Não houve retirada de excesso.
- **Recorte (comando do pesquisador).** 50 unigramas, ampliado até o desempate. A frequência do 50º é 25, e *objetivo* (25) entrou por empate, num total de 51. O primeiro fora é *iniciativa* (24).
- **Total.** Os 51 lemas somam 2.687 ocorrências, 32,6% dos 8.247 tokens de conteúdo.

#### Como ler
Barra = frequência absoluta; entre parênteses, a frequência por 1.000 tokens. A linha tracejada, quando há, marca o fim do *N* pedido; abaixo dela estão os lemas empatados. A tabela traz alcance e DP de cada lema.

#### Interpretação
**Observação.** As posições 1 a 31 são as da Visualização 2. Nas posições 32 a 51, com frequências de 32 a 25, aparecem:
- governança e responsabilidade: *governança*, *responsável*, *sustentável*;
- posição global: *global*;
- economia: *indústria*, *empresa*, *investimento*, *econômico*;
- sociedade: *social*, *sociedade*;
- o próprio plano: *pbia*, *plano*, *objetivo*;
- verbos de ação: *estabelecer*, *desenvolver*, *incluir*.

**Inferência.** O recorte ampliado acrescenta ao núcleo a governança responsável e sustentável, a dimensão econômica e a sociedade.

#### Fragilidades e ajustes possíveis
- *Pbia* (31) é a sigla do próprio documento, uma autorreferência. *Plano* e *objetivo* também se referem ao próprio texto.
- As frequências dessa faixa são próximas (25 a 32).
- *Sistema* (35) continua fora pela lista de *stopwords*.

### Visualização 6 — Os 75 principais unigramas
*Visualização construída por comando do pesquisador (2026-10-13): os 75 principais unigramas, com o recorte ampliado até o desempate quando houver empate no ponto de corte.*

In [ ]:
# Parâmetros desta seção (comando do pesquisador, 2026-10-13): os 75 principais unigramas, ampliado até o desempate
N_UNIGRAMAS_75 = 75
AMPLIAR_EMPATES_UNIGRAMAS_75 = True

TOP_UNIGRAMAS_75, INFO_UNIGRAMAS_75 = recorte(LEMAS, N_UNIGRAMAS_75, ampliar_empates=AMPLIAR_EMPATES_UNIGRAMAS_75, chave="lema")
titulo = f"Os {N_UNIGRAMAS_75} principais unigramas (lemas de conteúdo) — {ROTULO}"
if INFO_UNIGRAMAS_75["exibidos"] > N_UNIGRAMAS_75:
    titulo += f"\n{INFO_UNIGRAMAS_75['exibidos']} exibidos: recorte ampliado até o desempate"
barras_horizontais(TOP_UNIGRAMAS_75, "lema", "frequência", titulo, "frequência absoluta (entre parênteses: por 1.000 tokens)",
                   INFO_UNIGRAMAS_75, "unigramas_top75")

TAB_UNIGRAMAS_75 = TOP_UNIGRAMAS_75[["lema", "frequência", "por_mil", "alcance_unidades", "alcance_pct", "DP_norm",
                                     "retiradas_excesso"]].copy()
TAB_UNIGRAMAS_75["alcance_pct"] = 100 * TAB_UNIGRAMAS_75["alcance_pct"]
TAB_UNIGRAMAS_75.index = pd.RangeIndex(1, len(TAB_UNIGRAMAS_75) + 1, name="posição")
TAB_UNIGRAMAS_75 = TAB_UNIGRAMAS_75.rename(columns={"por_mil": "por 1.000 tokens", "alcance_unidades": "alcance (unidades)",
                                                     "alcance_pct": "alcance (%)", "DP_norm": "DP normalizado",
                                                     "retiradas_excesso": "retiradas (excesso marcado)"})
with pd.option_context("display.max_rows", None):
    display(tabela_br(TAB_UNIGRAMAS_75, {"por 1.000 tokens": 2, "alcance (%)": 1, "DP normalizado": 2}))

info = INFO_UNIGRAMAS_75
soma = int(TOP_UNIGRAMAS_75["frequência"].sum())
print("Síntese factual:")
print(f"- Recorte pedido: {N_UNIGRAMAS_75}; exibidos: {info['exibidos']}"
      + (f" (acrescentados por empate na frequência {br(int(info['limite']))}: {', '.join(info['acrescentados'])})"
         if info["acrescentados"] else " (sem empate no ponto de corte)")
      + (f"; primeiro lema fora: {info['primeiro_fora'][0]} ({br(int(info['primeiro_fora'][1]))})." if info["primeiro_fora"] else "."))
print(f"- Os {info['exibidos']} lemas somam {br(soma)} ocorrências ({br_pct(soma / len(CONT))} dos {br(len(CONT))} tokens de conteúdo); "
      f"frequência entre {br(int(TOP_UNIGRAMAS_75['frequência'].min()))} e {br(int(TOP_UNIGRAMAS_75['frequência'].max()))}.")
print("- Lista exibida (posição. lema: frequência): "
      + "; ".join(f"{i}. {r.lema}: {br(r.frequência)}" for i, r in enumerate(TOP_UNIGRAMAS_75.itertuples(), start=1)) + ".")
print("- Retirada do excesso marcado: " + (f"{br(int(TOP_UNIGRAMAS_75['retiradas_excesso'].sum()))} ocorrências retiradas nos lemas exibidos."
                                           if TOP_UNIGRAMAS_75["retiradas_excesso"].sum() else "nenhuma (este JSON não tem padrões marcados)."))


#### Método e escolhas
- **Medida.** As mesmas da Visualização 2, sem retirada de excesso.
- **Recorte (comando do pesquisador).** 75 unigramas, ampliado até o desempate. Não houve empate: o 75º é *educação* (20), e o primeiro fora é *aproveitar* (19).
- **Total.** Os 75 lemas somam 3.220 ocorrências, 39,0% dos 8.247 tokens de conteúdo.

#### Como ler
Barra = frequência absoluta; entre parênteses, a frequência por 1.000 tokens. A linha tracejada, quando há, marca o fim do *N* pedido; abaixo dela estão os lemas empatados. A tabela traz alcance e DP de cada lema.

#### Interpretação
**Observação.** Além dos 51 da Visualização 5, as posições 52 a 75 (frequências de 24 a 20) trazem:
- ética e benefícios: *ético*, *benefício*, *população*, *humano*;
- energia e computação: *energético*, *computacional*;
- setores de aplicação: *saúde*, *educação*;
- estrutura produtiva: *cadeia*, *ecossistema*;
- meios: *recurso*, *adoção*;
- termos de política: *política*, *iniciativa*, *eixo*.

**Inferência.** No recorte de 75 aparecem a infraestrutura energética e computacional, áreas de aplicação (saúde, educação) e a dimensão ética e de benefícios para a população.

#### Fragilidades e ajustes possíveis
- Com frequências de 20 a 24, diferenças de uma ou duas ocorrências reordenam a lista.
- *Eixo* (23) é palavra da organização do próprio plano.
- *Significativo* e *base* são genéricos.

### Visualização 7 — Os 50 principais bigramas
*Visualização construída por comando do pesquisador (2026-10-13): os 50 principais bigramas, com o recorte ampliado até o desempate quando houver empate no ponto de corte.*

In [ ]:
# Parâmetros desta seção (comando do pesquisador, 2026-10-13): os 50 principais bigramas, ampliado até o desempate
N_BIGRAMAS_50 = 50
AMPLIAR_EMPATES_BIGRAMAS_50 = True

TOP_BIGRAMAS_50, INFO_BIGRAMAS_50 = recorte(BIG_VALIDOS, N_BIGRAMAS_50, ampliar_empates=AMPLIAR_EMPATES_BIGRAMAS_50, chave="bigrama")
titulo = f"Os {N_BIGRAMAS_50} principais bigramas — {ROTULO}"
if INFO_BIGRAMAS_50["exibidos"] > N_BIGRAMAS_50:
    titulo += f"\n{INFO_BIGRAMAS_50['exibidos']} exibidos: recorte ampliado até o desempate"
barras_horizontais(TOP_BIGRAMAS_50, "bigrama", "frequência", titulo, "frequência absoluta (entre parênteses: por 1.000 tokens)",
                   INFO_BIGRAMAS_50, "bigramas_top50")

TAB_BIGRAMAS_50 = TOP_BIGRAMAS_50[["bigrama", "lemas", "frequência", "por_mil", "alcance_unidades", "retiradas_excesso",
                                   "formas_no_texto", "lemas_reunidos"]].copy()
TAB_BIGRAMAS_50.index = pd.RangeIndex(1, len(TAB_BIGRAMAS_50) + 1, name="posição")
TAB_BIGRAMAS_50 = TAB_BIGRAMAS_50.rename(columns={"bigrama": "bigrama (forma mais frequente no texto)", "lemas": "par de lemas",
                                                   "por_mil": "por 1.000 tokens", "alcance_unidades": "alcance (unidades)",
                                                   "retiradas_excesso": "retiradas (excesso marcado)", "formas_no_texto": "formas no texto",
                                                   "lemas_reunidos": "outros pares de lemas com a mesma forma escrita (reunidos)"})
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(tabela_br(TAB_BIGRAMAS_50, {"por 1.000 tokens": 2}))

info = INFO_BIGRAMAS_50
reun = TOP_BIGRAMAS_50[TOP_BIGRAMAS_50["lemas_reunidos"] != "–"]
print("Síntese factual:")
print(f"- {br(len(BIG_VALIDOS))} bigramas com frequência ≥ {FREQ_MINIMA_BIGRAMA}. Recorte pedido: {N_BIGRAMAS_50}; exibidos: {info['exibidos']}"
      + (f" (acrescentados por empate na frequência {br(int(info['limite']))}: {', '.join(info['acrescentados'])})"
         if info["acrescentados"] else " (sem empate no ponto de corte)")
      + (f"; primeiro bigrama fora: {info['primeiro_fora'][0]} ({br(int(info['primeiro_fora'][1]))})." if info["primeiro_fora"] else "."))
print("- Lista exibida (posição. bigrama: frequência): "
      + "; ".join(f"{i}. {r.bigrama}: {br(r.frequência)}" for i, r in enumerate(TOP_BIGRAMAS_50.itertuples(), start=1)) + ".")
print("- Pares de lemas reunidos por terem a mesma forma escrita (nos exibidos): "
      + ("; ".join(f"{r.bigrama} ← {r.lemas_reunidos}" for r in reun.itertuples()) if len(reun) else "nenhum") + ".")
print("- Retirada do excesso marcado: " + (f"{br(int(TOP_BIGRAMAS_50['retiradas_excesso'].sum()))} ocorrências retiradas nos bigramas exibidos."
                                           if TOP_BIGRAMAS_50["retiradas_excesso"].sum() else "nenhuma ocorrência retirada nos bigramas exibidos."))


#### Método e escolhas
- **Medida.** A mesma definição da Visualização 4, com frequência mínima 2 (268 bigramas). Não houve retirada de excesso.
- **Recorte (comando do pesquisador).** 50 bigramas, ampliado até o desempate. A frequência do 50º é 5, e dois empatados entraram (*soberania nacional*, *soberania tecnológica*), num total de 52. O primeiro fora é *américa latina* (4).
- **Formas reunidas.** *Ações propostas* e *vantagem competitiva*, como na Visualização 4.

#### Como ler
O rótulo é a forma escrita mais frequente do bigrama. A linha tracejada marca o fim dos 50 pedidos; abaixo dela estão os bigramas empatados. A tabela traz o par de lemas, as formas encontradas e o alcance.

#### Interpretação
**Observação.** As posições 1 a 35 são as da Visualização 4. As posições 36 a 52, todas com frequência 5, trazem:
- soberania e autonomia: *soberania nacional*, *soberania tecnológica*, *autonomia tecnológica*;
- energia: *energia renovável*, *energética predominantemente*, *predominantemente limpa*;
- regulação: *marco regulatório*, *propriedade intelectual*;
- dimensão nacional: *infraestrutura nacional*, *empresas brasileiras*, *indústria brasil*;
- outros: *inclusão social*, *instituições públicas*, *revolução tecnológica*, *larga escala*, *impacto imediato*, *segurança pública*.

**Inferência.** Soberania e autonomia tecnológica aparecem como expressões recorrentes (5 ocorrências cada), ao lado da energia limpa e do marco regulatório.

#### Fragilidades e ajustes possíveis
- *Energética predominantemente* e *predominantemente limpa* (5 cada) provavelmente vêm da mesma expressão repetida, *matriz energética predominantemente limpa*.
- *Indústria brasil* deve vir do nome *Nova Indústria Brasil*, já que *nova* é *stopword*.
- *Impacto imediato* vem de *ações de impacto imediato*.
- Esses três pontos são inferências a partir dos bigramas e das formas na tabela.
- Com o corte em 5, todos os bigramas acrescentados têm a mesma frequência, e a ordem entre eles é alfabética.

### Visualização 8 — Os 75 principais bigramas
*Visualização construída por comando do pesquisador (2026-10-13): os 75 principais bigramas, com o recorte ampliado até o desempate quando houver empate no ponto de corte. Seção acrescentada; as anteriores ficam intactas.*

In [ ]:
# Parâmetros desta seção (comando do pesquisador, 2026-10-13): os 75 principais bigramas, ampliado até o desempate
N_BIGRAMAS_75 = 75
AMPLIAR_EMPATES_BIGRAMAS_75 = True

TOP_BIGRAMAS_75, INFO_BIGRAMAS_75 = recorte(BIG_VALIDOS, N_BIGRAMAS_75, ampliar_empates=AMPLIAR_EMPATES_BIGRAMAS_75, chave="bigrama")
titulo = f"Os {N_BIGRAMAS_75} principais bigramas — {ROTULO}"
if INFO_BIGRAMAS_75["exibidos"] > N_BIGRAMAS_75:
    titulo += f"\n{INFO_BIGRAMAS_75['exibidos']} exibidos: recorte ampliado até o desempate"
barras_horizontais(TOP_BIGRAMAS_75, "bigrama", "frequência", titulo, "frequência absoluta (entre parênteses: por 1.000 tokens)",
                   INFO_BIGRAMAS_75, "bigramas_top75")

TAB_BIGRAMAS_75 = TOP_BIGRAMAS_75[["bigrama", "lemas", "frequência", "por_mil", "alcance_unidades", "retiradas_excesso",
                                   "formas_no_texto", "lemas_reunidos"]].copy()
TAB_BIGRAMAS_75.index = pd.RangeIndex(1, len(TAB_BIGRAMAS_75) + 1, name="posição")
TAB_BIGRAMAS_75 = TAB_BIGRAMAS_75.rename(columns={"bigrama": "bigrama (forma mais frequente no texto)", "lemas": "par de lemas",
                                                   "por_mil": "por 1.000 tokens", "alcance_unidades": "alcance (unidades)",
                                                   "retiradas_excesso": "retiradas (excesso marcado)", "formas_no_texto": "formas no texto",
                                                   "lemas_reunidos": "outros pares de lemas com a mesma forma escrita (reunidos)"})
with pd.option_context("display.max_rows", None, "display.max_colwidth", None):
    display(tabela_br(TAB_BIGRAMAS_75, {"por 1.000 tokens": 2}))

info = INFO_BIGRAMAS_75
reun = TOP_BIGRAMAS_75[TOP_BIGRAMAS_75["lemas_reunidos"] != "–"]
print("Síntese factual:")
print(f"- {br(len(BIG_VALIDOS))} bigramas com frequência ≥ {FREQ_MINIMA_BIGRAMA}. Recorte pedido: {N_BIGRAMAS_75}; exibidos: {info['exibidos']}"
      + (f" (acrescentados por empate na frequência {br(int(info['limite']))}: {', '.join(info['acrescentados'])})"
         if info["acrescentados"] else " (sem empate no ponto de corte)")
      + (f"; primeiro bigrama fora: {info['primeiro_fora'][0]} ({br(int(info['primeiro_fora'][1]))})." if info["primeiro_fora"] else "."))
print("- Lista exibida (posição. bigrama: frequência): "
      + "; ".join(f"{i}. {r.bigrama}: {br(r.frequência)}" for i, r in enumerate(TOP_BIGRAMAS_75.itertuples(), start=1)) + ".")
print("- Pares de lemas reunidos por terem a mesma forma escrita (nos exibidos): "
      + ("; ".join(f"{r.bigrama} ← {r.lemas_reunidos}" for r in reun.itertuples()) if len(reun) else "nenhum") + ".")
print("- Retirada do excesso marcado: " + (f"{br(int(TOP_BIGRAMAS_75['retiradas_excesso'].sum()))} ocorrências retiradas nos bigramas exibidos."
                                           if TOP_BIGRAMAS_75["retiradas_excesso"].sum() else "nenhuma ocorrência retirada nos bigramas exibidos."))


#### Método e escolhas
- **Medida.** A mesma definição da Visualização 4: pares de lemas de conteúdo adjacentes, na mesma frase, sem pontuação nem palavra retirada entre eles, com frequência mínima 2 (268 bigramas). Não houve retirada de excesso, porque o recorte sem anexos não tem padrões marcados.
- **Recorte (comando do pesquisador).** 75 bigramas, ampliado até o desempate. A frequência do 75º é 4, e seis empatados entraram (*processo regulatório*, *públicos federais*, *recursos naturais*, *segurança cibernética*, *setores industriais*, *união europeia*), num total de 81. O primeiro fora é *alinhamento estratégico* (3).
- **Formas reunidas.** *Ações propostas* e *vantagem competitiva*, como na Visualização 4.

#### Como ler
O rótulo é a forma escrita mais frequente do bigrama. A linha tracejada marca o fim dos 75 pedidos; abaixo dela estão os bigramas empatados. A tabela traz o par de lemas, as formas encontradas e o alcance.

#### Interpretação
**Observação.** As posições 1 a 52 são as da Visualização 7. As posições 53 a 81, todas com frequência 4, trazem:
- economia e indústria: *competitividade econômica*, *desenvolvimento econômico*, *indústria brasileira*, *setores industriais*, *gargalos específicos*;
- energia e recursos: *energética limpa*, *matriz elétrica*, *recursos naturais*;
- infraestrutura e capacidade: *capacidade computacional*, *infraestrutura digital*, *avanço tecnológico*, *dados nacionais*;
- sociedade, direitos e confiança: *bem-estar social*, *desafios sociais*, *direitos individuais*, *dados pessoais*, *confiança pública*, *inovação responsável*, *educação básica*;
- regulação e segurança: *processo regulatório*, *segurança cibernética*;
- relações externas: *cooperação internacional*, *américa latina*, *união europeia*.

**Inferência.** Na cauda da lista, ao lado do núcleo já visto, aparecem economia e indústria, energia, direitos e confiança e cooperação internacional, todos com a mesma frequência baixa.

#### Fragilidades e ajustes possíveis
- Os 29 bigramas das posições 53 a 81 têm frequência 4. O recorte pedido terminava no meio deles, e os 6 restantes entraram pelo empate. A ordem entre eles é alfabética e não indica importância.
- Nessa faixa há pares que são apenas vizinhança sintática, e não expressões fixas: *contribui significativamente*, *enfrenta desafios*, *mutuamente benéficas*.
- *Artificial intelligence* (4) é expressão em inglês citada no texto.
- *Energética limpa* (4), *energética predominantemente* e *predominantemente limpa* (5 cada) provavelmente vêm de variantes da mesma expressão sobre a matriz energética (inferência a partir das formas da tabela).